(c) Lisa M Dratva, April 2026
# Pan-infection atlas main figures and supplementary figures notebook

In [ ]:
from pathlib import Path
import os
import re
import json
import warnings
from typing import Optional, Union

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb
import matplotlib.gridspec as gridspec

from scipy import sparse
from scipy.stats import chisquare, chi2_contingency
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist, squareform

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)

import cell2tcr
from tcrdist.repertoire import TCRrep

# Adjust ROOT_DIR to the local location of the pan-infection atlas working directory.
ROOT_DIR = Path("/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection")
DATA_DIR = Path("data")
FIGURE_DIR = Path("results/figures")

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Keep these string aliases because the original analysis code used string concatenation.
root_dir = str(ROOT_DIR) + os.sep
savedir = str(FIGURE_DIR) + os.sep

plt.style.use("seaborn-v0_8-muted")
matplotlib.rcParams["savefig.transparent"] = True
matplotlib.rcParams["savefig.dpi"] = 300
matplotlib.rcParams["savefig.bbox"] = "tight"
matplotlib.rcParams["axes.spines.right"] = False
matplotlib.rcParams["axes.spines.top"] = False


## 1. Dataset overview

Load atlas metadata, harmonize pathogen labels used in the overview plot, and generate the study by pathogen bubble plot.

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 77
# load T cell atlas data
obs = pd.read_csv(
    root_dir + 'datasets/pan_infection_atlas/snakemake_toolbox/out/checkpoint_objects/checkpoint_6/obs.csv', 
    index_col=0,
)
obs.head()

# GSE259231 needs to be split. We got the data set through personal communication for GSE259231 (HBV) and GSE150305 (HCV = re-used data from a previous publication)
obs.loc[obs.pathogen=='HCV', 'study'] = 'GSE150305'

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 90
# mark non-healthy Reyes_2020 as Bacterial_infection
obs.loc[(obs.study=='Reyes_2020') & (obs.pathogen_type=='Bacterium'), 'pathogen'] = 'Bacterial_infection'

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 91
# mark non-pathogen Liu_2021 as healthy
obs.loc[(obs.study=='Liu_2021') & (obs.pathogen.isna()), 'pathogen'] = 'Healthy'

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 97

# mark other non-healthy controls
obs.loc[(obs.pathogen.isna()), 'pathogen'] = 'Other_control'

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 98
# split BEAM EBV-CMV-flu cells
beam_labels = pd.read_csv(root_dir+'datasets/pan_infection_atlas/miscellaneous/milo_proper/BEAM_pathogen_split_detail.csv', index_col=1)
beam_labels['pathogen'] = beam_labels.virus_detail.str.split('_', n=2, expand=True)[2].fillna('Influenza_CMV_EBV')
obs.loc[beam_labels.index, 'pathogen'] = beam_labels.pathogen.values

# split BEAM VZV-EBV cells 
obs.loc[obs.study=='GSE275633','pathogen'] = obs.loc[obs.study=='GSE275633', 'antigen'].apply(lambda x: 'EBV_VZV' if isinstance(x, float) else 'EBV' if 'EBV' in x else 'VZV' if 'VZV' in x else np.nan)


In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 103
buggert_antigen_specificity = pd.read_csv(root_dir+'datasets/pan_infection_atlas/annotations_LMD/GSE162097_HIV_CMV_antigen_specificity.csv')
buggert_antigen_specificity

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 104
buggert_to_pathogen_map = buggert_antigen_specificity[buggert_antigen_specificity.barcode.isin(obs.barcode)][['barcode','Pathogen_specificity']].set_index('barcode').Pathogen_specificity.to_dict()

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 106
# assign experimental specificities
obs.loc[obs.barcode.isin(buggert_to_pathogen_map.keys()), 'pathogen'] = obs.loc[obs.barcode.isin(buggert_to_pathogen_map.keys())].barcode.map(buggert_to_pathogen_map)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 107
# x_order = list(study_order)
x_order = ['GSE158769',
 'Ren_2021',
 'Stephenson_2021',
 'COMBAT_2022',
           
 'ESKD_2024',
 'Yoshida_2021',
 'Liu_2021',
 'Lindeboom_2024',
 'GSE287808',
           
 'Reyes_2020',
 'Waradon_Dengue',
 'Hatje_2024',
           
 'GSE182159',
 'GSE259231',
 'GSE234241',
 'GSE275633',
           
 'BEAM',
           
 'GSE162097',
 'HRA000190',
 'GSE187515',
 'GSE180268',
 'GSE173231',
 'GSE150305',
           
 'GSE217930',
 'GSE182536',
          ]
x_order

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 108
# y_order = list(pathogens)
y_order = ['Healthy',
 'Tuberculosis',
 'SARS-CoV-2',
 'Other_control',
 'Bacterial_infection',
 'Dengue_virus',
 'HBV',
 'VZV',
 'EBV',
 'Influenza_virus',
 'CMV',
 'HIV',
#  'EBV_VZV',
#  'Influenza_CMV_EBV',
 'HPV',
 'HSV-2',
 'HCV',
 'Plasmodium_falciparum',
          ]

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 109
# group your obs dataframe
df_plot = (
    obs.groupby(['study', 'pathogen'])
    .agg(
        donors=('donor_id', pd.Series.nunique),
        cells=('barcode', 'count')
    )
    .reset_index()
)

# optional: drop empty/blank pathogen labels (prevents weird extra row)
df_plot = df_plot[df_plot['pathogen'].notna() & (df_plot['pathogen'].astype(str).str.len() > 0)]

# scale donors explicitly into bubble sizes (areas)
scaler = MinMaxScaler((5, 250))
scaling_factor = 3
df_plot['donors_scaled'] = scaler.fit_transform(df_plot[['donors']]) * scaling_factor

# log color normalization for cells
norm = mcolors.LogNorm(vmin=df_plot['cells'].min(), vmax=df_plot['cells'].max())
cmap = plt.cm.inferno

# studies: largest donors on the left
study_order = (
    df_plot.groupby('study')['donors']
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

df_plot['study'] = pd.Categorical(df_plot['study'], categories=study_order, ordered=True)

# pathogens: sort by donors in the leftmost (largest) study, so biggest bubble is top-left
anchor_study = study_order[0]

pathogens_anchor = (
    df_plot[df_plot['study'] == anchor_study]
    .sort_values('donors', ascending=False)['pathogen']
    .tolist()
)

# fallback ordering for anything not present in anchor study
pathogens_global = (
    df_plot.groupby('pathogen')['donors']
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

# build final pathogen order: Healthy first, then anchor-based, then remaining global
seen = set(['Healthy'])
pathogens = ['Healthy']

for p in pathogens_anchor:
    if p not in seen and p != 'Healthy':
        pathogens.append(p)
        seen.add(p)

for p in pathogens_global:
    if p not in seen and p != 'Healthy':
        pathogens.append(p)
        seen.add(p)

# df_plot['pathogen'] = pd.Categorical(df_plot['pathogen'], categories=pathogens, ordered=True)



df_plot['study'] = pd.Categorical(df_plot['study'], x_order, ordered=True)
df_plot['pathogen'] = pd.Categorical(df_plot['pathogen'], y_order, ordered=True)

df_plot = df_plot[(df_plot.study.isin(x_order))&(df_plot.pathogen.isin(y_order))]

plt.figure(figsize=(10, 5))
scatter = plt.scatter(
    x=df_plot['study'].cat.codes,
    y=df_plot['pathogen'].cat.codes,
    s=df_plot['donors_scaled'],
    c=df_plot['cells'],
    cmap=cmap,
    norm=norm,
    edgecolor="k",
    linewidth=0.3,
)

# plt.xticks(ticks=range(len(study_order)), labels=study_order, rotation=90)
# plt.yticks(ticks=range(len(pathogens)), labels=pathogens)

plt.xticks(ticks=range(len(x_order)), labels=x_order, rotation=90)
plt.yticks(ticks=range(len(y_order)), labels=y_order)

plt.xlabel("Study")
plt.ylabel("Pathogen")
plt.title("Donor and cell number overview by study and pathogen")

# put Healthy (first category) at the top
plt.gca().invert_yaxis()

cbar = plt.colorbar(scatter, shrink=0.5, anchor=(0, -0.5))
cbar.set_label("Cells (n, log scale)")

for donors in [5, 10, 50, 100, 250]:
    size = scaler.transform([[donors]])[0][0] * scaling_factor
    plt.scatter([], [], s=size, c="gray", edgecolor="k", alpha=0.5, label=f"{donors}")
plt.legend(scatterpoints=1, frameon=False, labelspacing=1, title="Donors (n)", bbox_to_anchor=(1, 1))

plt.grid(True, color="grey", linewidth=0.2)
plt.tight_layout()
plt.savefig(savedir+'bubble_donor_cell_n_split_BEAM')
plt.savefig(savedir+'bubble_donor_cell_n_split_BEAM.pdf')
plt.show()


## 2. Public TCR motifs, invariant T cells, and HLA restriction

Generate public motif summaries, invariant TCR donor-count distributions, HLA donor summaries, and MHC restriction figures.

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 124
# match TCR processing as done for Cell2TCR
def clean_tcr(series: pd.Series) -> pd.Series:
    invalids = {"nan", "No_contig*01", "None*01", None, np.nan, "*01"}

    def _clean(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip()
        if val in invalids or val == "":
            return np.nan
        # if multiple genes, keep only the first
        if any(sep in val for sep in [",", ";", " "]):
            val = re.split(r"[ ,;]", val)[0]
        # normalize TCR genes
        if val.startswith(("TRAV", "TRBV", "TRAJ", "TRBJ")):
            if "*" not in val:
                val += "*01"
        return val
    
    return series.apply(_clean)
    

cols = ['IR_VDJ_1_j_call', 'IR_VDJ_1_v_call', 'IR_VJ_1_j_call', 'IR_VJ_1_v_call']
for col in cols:
    obs.loc[:,col] = clean_tcr(obs[col])

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 125
# load Cell2TCR results
df = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection_cell2tcr/cell2tcr_job/clone_df_with_motif_21.csv')
df.set_index('Unnamed: 0', inplace=True)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 126
# map TCR motifs to atlas
cols = ['IR_VDJ_1_j_call', 'IR_VDJ_1_junction_aa', 'IR_VDJ_1_v_call', 'IR_VJ_1_j_call', 'IR_VJ_1_junction_aa', 'IR_VJ_1_v_call']
# motif_dict = df.drop_duplicates(subset=cols).set_index(cols)['motif'].to_dict()
motif_dict = df[df.donor_id.notna()].drop_duplicates(subset=cols).set_index(cols)['motif'].to_dict()

# apply to full pan-infection t cell atlas object
obs['motif'] = obs[cols].apply(lambda row: motif_dict.get(tuple(row)), axis=1)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 127
cell2tcr.assign_column_names(obs)
obs.loc[:,'clone_id'] = obs.groupby(['donor_id','vdj_aa', 'vdj_v', 'vdj_j', 'vj_aa', 'vj_v', 'vj_j'], sort=False).ngroup()

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 128
# harmonize BEAM experiment annotations
cols = ['annotation_level_1', 'annotation_level_2', 'annotation_level_3', 'annotation_level_4']

# our own BEAM data
annot = pd.read_csv(root_dir+'datasets/pan_infection_atlas/annotations_LMD/BEAM_manual_annotation-2.csv')
reverse_barcode_dict = (obs[obs.study=='BEAM'].donor_id + '_' + obs[obs.study=='BEAM'].barcode.str.split('_|None', expand=True)[1]).to_dict()
annot.index = annot.Donor_barcodes.map({v: k for k, v in reverse_barcode_dict.items()})
annot = annot[annot.index.isin(obs.index)]
obs.loc[annot.index,cols] = annot.loc[:,cols].values

# VZV BEAM data
annot = pd.read_csv(root_dir + 'datasets/pan_infection_atlas/annotations_LMD/GSE275633_BEAM_annotation.csv')
annot.index = 'GSE275633' + annot['index']
barcode_map = obs[obs.study=='GSE275633'].reset_index().set_index('barcode')['index'].to_dict()
annot['new_barcode'] = annot.index.map(barcode_map)
obs.loc[annot.new_barcode, cols] = annot.loc[:,cols].values

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 130
def check_invariant(row):
    # Check for MAIT pattern
    if 'TRAV1-2*' in row['IR_VJ_1_v_call']:
        if any(substr in row['IR_VJ_1_j_call'] for substr in ['TRAJ33', 'TRAJ20', 'TRAJ12']):
            return 'MAIT'
    # Check for iNKT pattern
    elif 'TRAV10*' in row['IR_VJ_1_v_call']:
        if 'TRAJ18' in row['IR_VJ_1_j_call']:
            return 'iNKT'
    return 'Conventional'
df['invariant'] = df.apply(lambda row: check_invariant(row), axis=1)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 133
# inputs as 1D arrays
conventional_data = (
    df.loc[df.invariant == 'Conventional']
      .groupby('motif')['donor_id']
      .nunique()
      .to_numpy()
)

invariant_data = (
    df.loc[df.invariant != 'Conventional']
      .groupby('motif')['donor_id']
      .nunique()
      .to_numpy()
)


binrange = (0, 360)
bins = 30
edges = np.linspace(binrange[0], binrange[1], bins + 1)
widths = np.diff(edges)

# histogram counts
c_counts, _ = np.histogram(conventional_data, bins=edges)
i_counts, _ = np.histogram(invariant_data, bins=edges)

# pseudocount so zeros show on log axis (prevents gaps)
eps = 0.5
c = c_counts + eps
i = i_counts + eps

plt.figure(figsize=(5, 3))

plt.bar(edges[:-1], c, width=widths, align='edge', alpha=0.5, label='Conventional', linewidth=0.6, color='blue')
plt.bar(edges[:-1], i, width=widths, align='edge', alpha=0.5, label='MAIT and iNKT', linewidth=0.6, color='orange')

plt.yscale('log')
plt.ylim(0.5,200000)
plt.legend(title='T cell type')
plt.xlabel('Number of donors')
plt.ylabel('Number of TCR motifs (log)')
plt.title('Distribution of donors across TCR motifs')
plt.tight_layout()
plt.savefig(savedir+'distribution_donors_conventional_mait_inkt')
plt.savefig(savedir+'distribution_donors_conventional_mait_inkt.pdf')
plt.show()


In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 136
hla = pd.read_csv(root_dir+'datasets/pan_infection_atlas/miscellaneous/hla_combat_ren_lindeboom_eskd_beam.csv', index_col=0)
hla = hla[hla.index.isin(obs.donor_id)]
hla.shape

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 137
dataset_map = {
    'Ren':'Ren_2021',
    'Challenge':'Lindeboom_2024',
    'COMBAT':'COMBAT_2022',
    'ESKD_2024':'ESKD_2024',
    'BEAM':'BEAM',
}

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 138
hla.source.map(dataset_map).value_counts().plot.bar(ylabel='Number of donors \nwith resolved HLA haplotype', xlabel='', figsize=(1.5,3))
plt.grid(axis='y', alpha=.5)
plt.savefig(savedir+'donor_count_resolved_hla')
plt.savefig(savedir+'donor_count_resolved_hla.pdf')

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 139
hla['source'] = hla['source'].map(dataset_map)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 140
mhc_i = hla.loc[:,~hla.columns.str.contains('D')]
mhc_ii = hla.loc[:,hla.columns.str.contains('D|source')]
for ind, mhc_ in enumerate([mhc_i, mhc_ii]):
    mhc = mhc_.loc[:,mhc_.iloc[:,:-1].sum(axis=0).gt(11).to_frame(name='valid').query('valid==True').index.tolist()+['source']]
    color_dict = dict(zip(mhc.source.unique(), sns.palettes.color_palette(n_colors=len(mhc.source.unique()), palette='deep')))
    colors = pd.DataFrame(index=mhc.index)
    colors['study'] = mhc.source.map(color_dict)

    g = sns.clustermap(mhc.iloc[:,:-1], cmap='Blues', row_colors=colors, figsize=(15,12), cbar_pos=None)
    colors['study_name'] = hla.source.values
    legend_study = [mpatches.Patch(color=c, label=l) for c,l in colors[['study','study_name']].drop_duplicates().values]
    l_study = plt.legend(alignment='left', bbox_to_anchor=(0.4,.99), handles=legend_study, title='Study',prop={'size':14}, frameon=True, bbox_transform=plt.gcf().transFigure)
    plt.gca().add_artist(l_study)
#     g.ax_row_dendrogram.set_visible(False) #suppress row dendrogrm
    g.ax_col_dendrogram.set_visible(False) #suppress column dendrogram
    g.ax_heatmap.set_yticks([])
#     g.ax_cbar.set_visible(False)
#     plt.tight_layout()
    g.savefig(savedir+f'mhc_{ind+1}_genes_by_donors')
    g.savefig(savedir+f'mhc_{ind+1}_genes_by_donors.pdf')

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 143
value_columns = [
    'pathogen', 'pathogen_type', 'exposure', 'disease_stage', 'severity', 'days_since_disease_or_challenge', 'tissue',
    'annotation_level_1','annotation_level_2','annotation_level_3','annotation_level_4',#'isolation','modalities','Activated',#'Exhausted',
]
# assign patient metadata
df.loc[:,value_columns] = obs.loc[df.index, value_columns].values

col_list = list()
for col in ['pathogen']:#,'pathogen_type']:
    for val in df[col].unique():
        if val in ['HBV',np.nan,'HSV-2','Plasmodium_falciparum']: continue
        col_list.append(f'{val}')
#         df.loc[:,f'{col}_{val}'] = df.loc[:,col] == val
        df.loc[:,val] = df.loc[:,col] == val

df['is_CD8'] = df.annotation_level_1 == 'T CD8'
df['is_CD4'] = df.annotation_level_1 == 'T CD4'
df['is_MAIT'] = df.annotation_level_1 == 'T MAI'

df_motif = df.groupby('motif')[col_list+['is_CD8','is_CD4','is_MAIT']].mean()
df_motif[['n_clones','n_donors','n_studies']] = df.groupby('motif')[['barcode','donor_id','study']].nunique()

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 144
obs_ = obs[obs.donor_id.isin(hla.index)] # must execute after having counted total number of donors

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 145
n_donors = 2 # 3
df_motifs = df_motif[df_motif.n_donors.ge(n_donors)]
df_motifs['motif'] = df_motifs.index.values
df_motifs.shape

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 146
# subset to MHC-I and MHC-II
hla_mhc1 = hla.loc[:,~hla.columns.str.contains('D')]
hla_mhc2 = hla.loc[:,hla.columns.str.contains('D|source')]
hla_mhc1 = hla_mhc1.iloc[:,:-1]
hla_mhc2 = hla_mhc2.iloc[:,:-1]

def get_majority_allele(motif_nr):
    individuals = obs_[obs_.motif==motif_nr].donor_id.unique()
    MHC_I = hla_mhc1.loc[individuals].mean(axis=0).sort_values()[-1:]
    MHC_II = hla_mhc2.loc[individuals].mean(axis=0).sort_values()[-1:]
    return [*MHC_I.index, *MHC_I.values, *MHC_II.index, *MHC_II.values]

def get_all_majority_alleles(motif_nr):
    individuals = obs_[obs_.motif==motif_nr].donor_id.unique()
    MHC_I = hla_mhc1.loc[individuals].mean(axis=0).to_frame(name='prop').query('prop==1.').index.tolist()
    MHC_II = hla_mhc2.loc[individuals].mean(axis=0).to_frame(name='prop').query('prop==1.').index.tolist()
    return MHC_I, MHC_II

# provide the majority allele for both MHC-I and MHC-II
df_motifs.loc[:,['MHC_I_allele', 'MHC_I_proportion','MHC_II_allele', 'MHC_II_proportion']] = df_motifs.apply(lambda x: get_majority_allele(x.motif), axis=1, result_type='expand').values
df_motifs.loc[:,['MHC_I_all', 'MHC_II_all']] = df_motifs.apply(lambda x: get_all_majority_alleles(x.motif), axis=1, result_type='expand').values

# True MHC-II restriction : either shared DRB, or shared DQA+DQB, or shared DPA+DPB
def is_mhc_ii_restricted(mhc_ii_list):
    if mhc_ii_list:
        if 'DRB' in '\t'.join(mhc_ii_list):
            return True
        elif 'DQA' in '\t'.join(mhc_ii_list) and 'DQB' in '\t'.join(mhc_ii_list):
            return True
        elif 'DPA' in '\t'.join(mhc_ii_list) and 'DPB' in '\t'.join(mhc_ii_list):
            return True
        else: return False
    else: return False

df_motifs['MHC_I_restricted'] = df_motifs.MHC_I_all.str.len().ge(1)
df_motifs['MHC_II_restricted'] = df_motifs.MHC_II_all.apply(is_mhc_ii_restricted)

def return_mhc_i_restricted_allele(mhc_list):
    if mhc_list:
        if len(mhc_list) == 1:
            return mhc_list[0]
        else:
            return 'Multiple'
    else:
        return 'Not restricted'
    
def return_mhc_ii_restricted_allele(mhc_list):
    if mhc_list:
        if len(mhc_list) == 1:
            if 'DRB' in mhc_list[0]:
                return mhc_list[0]
            else:
                return 'Not restricted'
        else: # check which allele combination is present
            count = pd.DataFrame(columns=['DRB', 'DPA', 'DPB', 'DQA', 'DQB'], data=np.zeros((1,5))) # count all mhcs
            for i in mhc_list:
                for allele in ['DRB', 'DPA', 'DPB', 'DQA', 'DQB']:
                    if allele in i:
                        count[allele] += 1
            count_comp = pd.DataFrame(columns=['DRB', 'DP', 'DQ'], data=np.zeros((1,3))) # counts complete mhc loci
            count_comp['DRB'] = count['DRB']
            count_comp['DP'] = count[['DPA','DPB']].all(axis=1)*count[['DPA','DPB']].sum(axis=1)/2 # check for presence of both chains
            count_comp['DQ'] = count[['DQA','DQB']].all(axis=1)*count[['DQA','DQB']].sum(axis=1)/2 # check for presence of both chains
            if count_comp.sum(axis=1).values > 1:
                return 'Multiple'
            elif count_comp.sum(axis=1).values < 1:
                return 'Not restricted'
            elif count_comp.sum(axis=1).values == 1: # should only equal 1 if full mhc molecule is presence (drb, or dqa+dqb, or dpa+dpb)
                allele_res = count_comp.T.rename(columns={0:'col'}).query('col==1').index.tolist()[0] # this is the restricted allele
                return_str = ''
                for i in sorted(mhc_list):
                    if allele_res in i:
                        if return_str:
                            return_str += '-'+i
                        else:
                            return_str += i
                return return_str
    else:
        return 'Not restricted'

df_motifs['MHC_I_restricted_allele'] = df_motifs['MHC_I_all'].apply(return_mhc_i_restricted_allele)
df_motifs['MHC_II_restricted_allele'] = df_motifs['MHC_II_all'].apply(return_mhc_ii_restricted_allele)
df_motifs

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 147
df_motifs.to_csv('data/df_motifs_with_hla.csv')

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 152
n_ind = 4 # restrict analysis to TCR motifs shared among n_ind or more donors
df_sub_ = df_motifs[(df_motifs.is_CD8>thresh_cd8)&(df_motifs.n_donors.ge(n_ind))&(df_motifs.is_MAIT.lt(0.2))]
# df_sub_ = m_sub[m_sub.is_CD8_bool] #df_motifs[(df_motifs.is_CD8>0.5)&(df_motifs.n_donors.ge(n_ind))&(df_motifs.is_MAIT.lt(0.2))]
df_sub = df_sub_[~df_sub_.MHC_I_restricted_allele.isin(['Not restricted','Multiple'])]
n_not_restricted = df_sub_.MHC_I_restricted_allele.eq('Not restricted').sum()
n_multiple = df_sub_.MHC_I_restricted_allele.eq('Multiple').sum()
df_sub.MHC_I_restricted_allele.value_counts().plot.pie(
    ylabel='', title=f'HLA class 1 restriction of {df_sub_.shape[0]} CD8+ TCR motifs shared among {n_ind} or more donors\n({n_not_restricted} non-restricted and {n_multiple} multiple restricted TCR motifs not shown)',
    autopct=lambda x: '{:.0f}'.format(x * (df_motifs[(df_motifs.is_CD8>thresh_cd8)&(df_motifs.n_donors.ge(n_ind))].MHC_I_restricted_allele.count()) / 100),
    figsize=[6,6], #cmap='cool',
)
plt.savefig(savedir+f'4_digit_mhc_i_restriction_pie_nind_{n_ind}')
plt.savefig(savedir+f'4_digit_mhc_i_restriction_pie_nind_{n_ind}.pdf')
plt.show()

df_sub.MHC_I_restricted_allele.value_counts(ascending=True).plot.barh(
    ylabel='', 
    title=f'HLA class 1 restriction of shared CD8+ TCR motifs\n({n_not_restricted} non-restricted and {n_multiple} multiple restricted TCR motifs not shown)', 
    figsize=(4,8), #cmap='cool',
)
plt.savefig(savedir+f'4_digit_mhc_i_restriction_list_nind_{n_ind}')
plt.savefig(savedir+f'4_digit_mhc_i_restriction_list_nind_{n_ind}.pdf')
plt.show()

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 153
n_ind = 4 # restrict analysis to TCR motifs shared among n_ind or more donors
df_sub_ = df_motifs[(df_motifs.is_CD4>thresh_cd4)&(df_motifs.n_donors.ge(n_ind))&(df_motifs.is_MAIT.lt(0.2))]
print(df_sub_.shape)
df_sub = df_sub_[~df_sub_.MHC_II_restricted_allele.isin(['Not restricted','Multiple'])]
n_not_restricted = df_sub_.MHC_II_restricted_allele.eq('Not restricted').sum()
n_multiple = df_sub_.MHC_II_restricted_allele.eq('Multiple').sum()
df_sub.MHC_II_restricted_allele.value_counts().plot.pie(
    ylabel='', title=f'HLA class 2 restriction of {df_sub_.shape[0]} CD4+ TCR motifs shared among {n_ind} or more donors\n({n_not_restricted} non-restricted and {n_multiple} multiple restricted TCR motifs not shown)',
    autopct=lambda x: '{:.0f}'.format(x * (df_motifs[(df_motifs.is_CD4>thresh_cd4)&(df_motifs.n_donors.ge(n_ind))].MHC_II_restricted_allele.count()) / 100),
    figsize=[5,5])
plt.savefig(savedir+f'4_digit_mhc_ii_restriction_pie_nind_{n_ind}')
plt.savefig(savedir+f'4_digit_mhc_ii_restriction_pie_nind_{n_ind}.pdf')
plt.show()

df_sub.MHC_II_restricted_allele.value_counts(ascending=True).plot.barh(
    ylabel='', title=f'HLA class 2 restriction of shared CD4+ TCR motifs\n({n_not_restricted} non-restricted and {n_multiple} multiple restricted TCR motifs not shown)', figsize=(4,8)
)
plt.savefig(savedir+f'4_digit_mhc_ii_restriction_list_nind_{n_ind}')
plt.savefig(savedir+f'4_digit_mhc_ii_restriction_list_nind_{n_ind}.pdf')
plt.show()

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 155
def get_all_majority_alleles_indiv(individuals):
    MHC_I = hla_mhc1.loc[individuals].mean(axis=0).to_frame(name='prop').query('prop==1.').index.tolist()
    MHC_II = hla_mhc2.loc[individuals].mean(axis=0).to_frame(name='prop').query('prop==1.').index.tolist()
#     print('MHC1:',len(MHC_I)>0, 'MHC2:', is_mhc_ii_restricted(MHC_II))
    return len(MHC_I)>0, is_mhc_ii_restricted(MHC_II)
#     return MHC_I, MHC_II

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 157
# this was useful as we compare 10 motifs at a time, and average the results n_rep times. 
n_motif = 6
n_inds = 8
n_rep = 8

df = pd.DataFrame(columns=['n','rep','MHC1_Random','MHC1_TCR motif','MHC2_Random','MHC2_TCR motif', 'CD'])
for sign, CD in zip(['<','>'], ['CD4', 'CD8']): # collect info on CD4 and CD8 motifs
    for rep in range(n_rep):
        for n_ind in range(2,n_inds+1):
            mhc1 = 0
            mhc2 = 0
            for i in range(n_motif): # build random motif composition
                indiv_list = hla.sample(n_ind).index.tolist()
            #     print(indiv_list)
                mhc1_, mhc2_ = get_all_majority_alleles_indiv(indiv_list)
                mhc1 += mhc1_
                mhc2 += mhc2_
            mhc1_c = mhc1/n_motif
            mhc2_c = mhc2/n_motif
            mhc1_m, mhc2_m = df_motifs.query('(n_donors == @n_ind)&(is_CD8'+sign+'.5)').sample(n_motif)[['MHC_I_restricted','MHC_II_restricted']].mean()
            df = pd.concat([df, pd.DataFrame(data=[[n_ind, rep, mhc1_c, mhc1_m, mhc2_c, mhc2_m, CD]],columns=['n','rep','MHC1_Random','MHC1_TCR motif','MHC2_Random','MHC2_TCR motif', 'CD'])], axis=0)

df_ = df.melt(id_vars=['n','rep','CD'])#.variable.str.split('_',expand=True)
df_ = pd.concat([df_,df_.variable.str.split('_',expand=True)], axis=1)
df_.columns = df_.columns[:4].tolist() + ['Restricted', 'MHC', 'Donor source']

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 158
# # Error bars represent n_rep iterations
# fig, ax = plt.subplots(ncols=2, sharey=True, figsize=(6,3))
# sns.pointplot(data=df_[df_.MHC=='MHC1'], x='n', y='Restricted', hue='Donor source', ax=ax[0])
# sns.pointplot(data=df_[df_.MHC=='MHC2'], x='n', y='Restricted', hue='Donor source', ax=ax[1])
# ax[0].set_title('MHC-I (all motifs)')
# ax[1].set_title('MHC-II (all motifs)')
# plt.tight_layout()
# plt.suptitle('Average MHC restriction across n donors', y=1.1)
# # plt.savefig(savedir+'mhc_across_n_donors')

# Error bars represent n_rep iterations
fig, ax = plt.subplots(ncols=2, sharey=True, figsize=(7,3))
sns.pointplot(data=df_[(df_.MHC=='MHC1')&(df_.CD=='CD8')], x='n', y='Restricted', hue='Donor source', ax=ax[0])
sns.pointplot(data=df_[(df_.MHC=='MHC2')&(df_.CD=='CD4')], x='n', y='Restricted', hue='Donor source', ax=ax[1])
sns.move_legend(ax[1], loc='right', bbox_to_anchor=(1.6,0.6))
ax[0].set_title('MHC-I  (CD8+ TCR motifs)')
ax[1].set_title('MHC-II (CD4+ TCR motifs)')
ax[0].legend().remove()
# ax[1].legend(bbox_inches=(1,1))
plt.tight_layout()
ax[0].set_ylabel('Proportion of donors\nwith shared MHC genes')
ax[1].set_ylabel('')
plt.suptitle('Average MHC restriction across n donors', y=1.1)
plt.savefig(savedir+'mhc_across_n_donors_cd4_cd8')
plt.savefig(savedir+'mhc_across_n_donors_cd4_cd8.pdf')

## 3. TCR similarity clustermap and VDJdb annotation

Recompute the selected motif distance matrix and generate the final clustermap with predicted pathogen and VDJdb annotation colors.

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 160
# recreate the Cell2TCR object
df = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection/datasets/pan_infection_atlas/snakemake_toolbox/out/checkpoint_objects/checkpoint_6/obs.csv')
 
print('Loaded data.')

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 161
# TCRdist compatible genes
genes = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/repos_hpc/cell2tcr/cell2tcr/alphabeta_gammadelta_db.tsv', sep='\t')
genes = genes[(genes.organism=='human')]
genes = genes[((genes.id.str.contains('A'))&(genes.chain=='A'))|((genes.id.str.contains('B'))&(genes.chain=='B'))]
genes = genes[~genes.id.str.contains('G|D')]

def clean_tcr(series: pd.Series) -> pd.Series:
    invalids = {"nan", "No_contig*01", "None*01", None, np.nan, "*01"}

    def _clean(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip()
        if val in invalids or val == "":
            return np.nan
        # if multiple genes, keep only the first
        if any(sep in val for sep in [",", ";", " "]):
            val = re.split(r"[ ,;]", val)[0]
        # normalize TCR genes
        if val.startswith(("TRAV", "TRBV", "TRAJ", "TRBJ")):
            if "*" not in val:
                val += "*01"
        return val
    
    return series.apply(_clean)
    

cols = ['IR_VDJ_1_j_call', 'IR_VDJ_1_v_call', 'IR_VJ_1_j_call', 'IR_VJ_1_v_call']
for col in cols:
    df.loc[:,col] = clean_tcr(df[col])
    # remove nan / None
    df[col] = df[col].fillna("").astype(str)
    df = df[df[col] != ""]
    
    # TCRdist compatibility
    # coerce unknown alleles >*01 to *01
    mask_bad = ~df[col].isin(genes.id.values)
    df.loc[mask_bad, col] = df.loc[mask_bad, col].str.replace(r"\*\d+$", "*01", regex=True)

    # drop anything still not recognized
    df = df[df[col].isin(genes.id.values)]
    print(col, 'Problematic:', [i for i in df[col].unique() if i not in genes.id.values])

cell2tcr.assign_column_names(df)

# only allow TCR-a and TCR-b genes
mask = (
    df['vdj_v'].astype(str).str.contains('B', na=False) &
    df['vdj_j'].astype(str).str.contains('B', na=False) &
    df['vj_v'].astype(str).str.contains('A', na=False) &
    df['vj_j'].astype(str).str.contains('A', na=False)
)
df = df[mask]

print('Modified or removed problematic VDJ.')
print(df.shape)

df.loc[:,'clone_id'] = df.groupby(['donor_id','vdj_aa', 'vdj_v', 'vdj_j', 'vj_aa', 'vj_v', 'vj_j'], sort=False).ngroup()
new_cols = {'cdr3_a_aa':'vj_aa', 'cdr3_b_aa':'vdj_aa', 'v_b_gene':'vdj_v', 'j_b_gene':'vdj_j', 'v_a_gene':'vj_v', 'j_a_gene':'vj_j',}

# make tcrdist-compatible
df.rename(columns = {v: k for k, v in new_cols.items()}, inplace=True)
df.drop(columns=df.columns[df.columns.duplicated()], inplace=True)    

# harmonize BEAM experiment annotations
cols = ['annotation_level_1', 'annotation_level_2', 'annotation_level_3', 'annotation_level_4']

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 162
# our own BEAM data
annot = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection/datasets/pan_infection_atlas/annotations_LMD/BEAM_manual_annotation-2.csv')
reverse_barcode_dict = (df[df.study=='BEAM'].donor_id + '_' + df[df.study=='BEAM'].barcode.str.split('_|None', expand=True)[1]).to_dict()
annot.index = annot.Donor_barcodes.map({v: k for k, v in reverse_barcode_dict.items()})
annot = annot[annot.index.isin(df.index)]
df.loc[annot.index,cols] = annot.loc[:,cols].values

# VZV BEAM data
annot = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection/datasets/pan_infection_atlas/annotations_LMD/GSE275633_BEAM_annotation.csv')
annot.index = 'GSE275633' + annot['index']
barcode_map = df[df.study=='GSE275633'].reset_index().set_index('barcode')['index'].to_dict()
annot['new_barcode'] = annot.index.map(barcode_map)
annot = annot[annot.new_barcode.isin(df.index)]
df.loc[annot.new_barcode, cols] = annot.loc[:,cols].values

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 176
df_m = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection_cell2tcr/cell2tcr_job/clone_df_with_motif_21.csv')#, index_col='Unnamed: 0')

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 178
motif_barcodes = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection_cell2tcr/cell2tcr_job/clone_df_with_motif_21_barcodes.csv', index_col=0)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 179
df['precomputed_motif'] = df['Unnamed: 0'].map(motif_barcodes.motif.to_dict())
df_dedup = df.drop_duplicates(subset='clone_id')
df_dedup['motif'] = pd.read_csv("/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection_cell2tcr/cell2tcr_job/membership_partition_21.csv", index_col=0)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 182
motifs_of_interest = [25,32,35,36,37,40,41,46,47,49,53,57,60,71,82,84,85,97,104,123,131,161,170,180,194,216,234,239]

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 185
df_c2t = df_dedup[df_dedup.precomputed_motif.isin(motifs_of_interest)].copy()
df_c2t.shape

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 187
cell2tcr.assign_column_names(df_c2t)
tr = cell2tcr.motifs(df_c2t, sparse=False, add_suffix=False, return_distances=True, chunk_size=300)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 188
predicted_pathogen_map = {
    25: "Unresolved",
    32: "EBV",
    35: "EBV",
    36: "EBV",
    37: "CMV",
    40: "SARS-CoV-2",
    41: "SARS-CoV-2",
    42: "SARS-CoV-2",
    48: "SARS-CoV-2",
    46: "Dengue",
    47: "Influenza",
    49: "Influenza",
    53: "CMV",
    57: "CMV",
    60: "HPV",
    61: "HPV",
#     71: "Stephenson-only 27 donor motif?",
    82: "HIV",
    84: "VZV",
    85: "Influenza",
    97: "HIV",
    104: "VZV",
    123: "HIV",
#     (131, 148): "HIV, HBV",
    161: "HPV",
#     170: "Multi-viral?",
#     180: "HIV/HBV CD4",
#     194: "Multi-viral HIV/malaria/dengue CD4",
    216: "HIV", # CD4
#     234: "HBV multi-viral",
#     239: "HSV / dengue /",
}


In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 194
beta_scores = cell2tcr.db_match(tr.clone_df.cdr3_b_aa)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 196
cell2tcr.db_annotate(tr.clone_df, beta_scores, 'cdr3_b_aa')

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 197
vdjdb_pathogen_map = {
    # Coronaviruses
    "SARS-CoV2": "SARS-CoV-2",
    "SARS coronavirus BJ01": "SARS-CoV-2",
    "SARS coronavirus Urbani (SARS-CoV (Urbani strain))": "SARS-CoV-2",
    "SARS-CoV1": "SARS-CoV-2",

    # EBV
    "Human herpesvirus 4 (Epstein Barr virus)": "EBV",
    "Human herpesvirus 4 strain B95-8 (Epstein-Barr virus (strain B95-8))": "EBV",

    # CMV
    "Human herpesvirus 5 (Human cytomegalovirus)": "CMV",
    "Human herpesvirus 5 strain AD169 (Human cytomegalovirus (strain AD169))": "CMV",
    "Human herpesvirus 5 strain Towne (Human cytomegalovirus (strain Towne))": "CMV",

    # Influenza
    "Influenza A virus": "Influenza_virus",
    "Influenza A virus (A/Luxembourg/46/2009(H1N1))": "Influenza_virus",
    "Influenza A virus H3N2 (A/Resvir-9 (H3N2))": "Influenza_virus",
    "Influenza A virus (A/Puerto Rico/8/1934(H1N1)) (Influenza A virus (A/PR 8/34 (H1N1)))": "Influenza_virus",

    # Hepatitis
    "Hepatitis C virus": "HCV",
    "hepatitis C virus genotype 1a (Hepatitis C virus subtype 1a)": "HCV",
    "Hepatitis B virus (Human hepatitis B virus)": "HBV",

    # HIV
    "Human immunodeficiency virus 1 (human immunodeficiency virus 1 HIV-1)": "HIV",

    # Other viruses already seen but not in your motif dict
    "Yellow fever virus 17D (Yellow fever virus (STRAIN 17D))": "Other",
    "Lymphocytic choriomeningitis mammarenavirus (Lymphocytic choriomeningitis virus)": "Other",
    "Murine hepatitis virus strain JHM (Murine coronavirus mhv (STRAIN JHM))": "Other",
    "Human respiratory syncytial virus A2 (Human respiratory syncytial virus (strain A2))": "Other",

    # Bacteria
    "Escherichia coli": "Other",

    # Non-human / irrelevant antigens
    "Homo sapiens (human)": "Other",
    "Gallus gallus (chicken)": "Other",
    "Triticum aestivum (Canadian hard winter wheat)": "Other",
}


In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 199
vdjdb_pathogen_palette = {
    # present in the image legend (exact same colors)
    "CMV": "tab:blue",         # dark blue
    "EBV": "tab:orange",         # orange
    "HIV": "#EFBD82",         # light orange
    "HPV": "tab:green",         # green
    "Influenza_virus": "yellowgreen",   # light green
    "SARS-CoV-2": "tab:red",  # red
    "VZV": "#8D6CB8",         # purple
#     "Unresolved": "#EB9E99",  # pink

    # new categories not in the image
    "HBV": "tab:cyan",         # teal-blue (distinct from CMV)
#     "HCV": "#C77C02",         # darker amber (distinct from EBV/HIV)
    "Other": "lightgrey",       # neutral grey fallback
    
}

from matplotlib.colors import ListedColormap
order = [
    "CMV", "EBV", "HIV", "HBV", #"HCV",
    "HPV", "Influenza_virus", "SARS-CoV-2", "VZV",
    "Other", 
]
cmap = ListedColormap(
    [vdjdb_pathogen_palette[k] for k in order],
#     name="predicted_pathogen"
)

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 200
tr_sub = tr
idx = tr_sub.clone_df.index
tr_sub.clone_df['predicted_pathogen'] = tr_sub.clone_df.precomputed_motif.map(predicted_pathogen_map)
tr_sub.clone_df['vdjdb_pathogen'] = tr_sub.clone_df.organism.map(vdjdb_pathogen_map)

# row colors
colors = pd.DataFrame(index=idx)
for col in ['donor_id','study',
#             'pathogen',
            'motif','predicted_pathogen']:
    tr_sub.clone_df[col] = tr_sub.clone_df[col].astype(str)
    lut = dict(zip(
        sorted(tr_sub.clone_df[col].unique()), 
        sns.palettes.color_palette(n_colors=len(tr_sub.clone_df[col].unique()),palette='tab20')))
    colors[col] = tr_sub.clone_df[col].map(lut)
colors['vdjdb_pathogen'] = tr_sub.clone_df['vdjdb_pathogen'].map(vdjdb_pathogen_palette)
    
g = sns.clustermap(
    pd.DataFrame(tr_sub.pw_alpha + tr_sub.pw_beta, index=idx),
    method='ward',
    cmap='RdBu', 
    cbar_kws={'label':'TCR distance'},
    row_colors=colors,
)
g.ax_heatmap.set(xticklabels=[], xticks=[], yticklabels=[], yticks=[])
g.ax_row_dendrogram.set_visible(False) #suppress row dendrogrm
g.ax_col_dendrogram.set_visible(False) #suppress column dendrogram

# legend for row colors
for ind, col in enumerate(['study','predicted_pathogen','vdjdb_pathogen']):
    colors[f'{col}_name'] = tr_sub.clone_df[col].values
#     legend = [mpatches.Patch(color=c, label=l) for c,l in colors[[col,f'{col}_name']].drop_duplicates().sort_values(f'{col}_name').values if not l == 'nan']
    legend = [
        mpatches.Patch(color=c, label=l)
        for c, l in (
            colors[[col, f'{col}_name']]
            .dropna(subset=[col, f'{col}_name'])   # drop NaN colors or labels
            .drop_duplicates()
            .sort_values(f'{col}_name')
            .values
        )
    ]
    legend_ = plt.legend(loc='center left', alignment='left', bbox_to_anchor=(0.01,0.2+ind*0.25), handles=legend, title=col, prop={'size':8}, frameon=True, bbox_transform=plt.gcf().transFigure)
    plt.gca().add_artist(legend_)
# plt.gca().add_artist(
#     plt.legend(
#         loc='center left', alignment='left', bbox_to_anchor=(0.01,0.2+2*0.25), handles=legend, 
#         title='VDJdb match', prop={'size':8}, frameon=True, bbox_transform=plt.gcf().transFigure))
g.savefig(savedir+'clustermap_with_vdjdb')
g.savefig(savedir+'clustermap_with_vdjdb.pdf')
plt.show()

## 4. Multi-Milo differential abundance plots

Define the polished Multi-Milo plotting function and run the saved pathogen-versus-healthy comparisons.

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 224
tissue_colors = {
    "BALF": "#1f77b4",
    "Cervix": "#ff7f0e",
    "Liver": "#2ca02c",
    "NALT": "#d62728",
    "Nasopharyngeal swabs": "#9467bd",
    "PBMCs": "#c49c94",
    "Sputum": "#f7b6d2",
    "TDL": "#c7c7c7",
    "Tumour": "#dbdb8d",
    "metastatic_lymphnode": "#9edae5",
}
pathogen_colors = {
    "CMV": "#393b79",
    "Dengue_virus": "#5254a3",
    "EBV": "#6b6ecf",
    "HBV": "#637939",
    "HCV": "#8ca252",
    "HIV": "#b5cf6b",
    "HPV": "#8c6d31",
#     "HSV-2": "#bd9e39",
    "HSV_2": "#bd9e39",
    "Healthy": "#e7ba52",
    "Influenza_CMV_EBV": "#843c39",
    "Influenza_virus": "#ad494a",
    "Plasmodium_falciparum": "#d6616b",
#     "SARS-CoV-2": "#7b4173",
    "SARS_CoV_2": "#7b4173",
    "Tuberculosis": "#a55194",
    "VZV": "#ce6dbd",
#     "Other_Control": "#de9ed6",
}

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 225
def plot_multi_milo(
    compare_variable: str,
    annotation: str,
    milo_dir: Union[str, Path],
    exp_df: Union[str, Path, pd.DataFrame],
    group_of_interest: Optional[str] = None,
    only_show_group_of_interest: bool = False,
    compartment: str = "CD4",
    savedir: Optional[Union[str, Path]] = None,
    significant_only: bool = True,
    sig_thresh: float = 0.20,
    sig_thresh_strict: float = 0.10,
    sig_star: bool = False,
    blood_only: bool = False,
    ignore_groups: Optional[Sequence[str]] = None,
    add_filter: Optional[Tuple[str, str]] = None,
    drop_duplicates: bool = False,
    df_filter_exclude_annotation: Sequence[str] = ("Mixed",),
#     tissue_colors: Optional[Dict[str, str]] = None,
    group_palette: Optional[Sequence] = None,
    cmap: str = "bwr",
    center: float = 0.0,
    col_cluster: bool = True,
    row_cluster: bool = True,
    figure_size: Tuple[int, int] = None,
    return_objects: bool = True,
) -> Optional[Tuple[sns.matrix.ClusterGrid, pd.DataFrame, pd.DataFrame]]:
    """
    Plot a Milo-style clustermap of per-experiment cell-state enrichment/depletion.

    This function:
      1) Reads per-experiment Milo summary CSVs (one per experiment ID) from
         `<milo_dir>/<annotation>/results_{compartment}_{id}.csv`.
      2) Optionally filters experiments (e.g., PBMCs-only; restrict to a given group; ignore some groups).
      3) Aligns the sign of logFC such that `group_of_interest` is consistently the numerator:
         - If `group_of_interest == "Healthy"`, it is forced into Group 2; logFC is flipped when Healthy is Group 1.
         - Else, `group_of_interest` is forced into Group 1; logFC is flipped when it appears in Group 2.
      4) Optionally keeps only significant neighbourhoods (SpatialFDR < `sig_thresh`) and drops specified annotations.
      5) Builds a matrix (experiments × nhood_annotation) of logFC values and plots a clustered heatmap with row color bars
         indicating tissue and experimental groups.
      6) Optionally overlays significance marks (“*” for `SpatialFDR < sig_thresh_strict`, “°” for `SpatialFDR < sig_thresh`).

    Parameters
    ----------
    compare_variable : str
        The column in the experiment metadata that defines the comparison (e.g., "severity", "status").
    annotation : str
        Annotation level used in Milo summary files (part of the folder path).
    milo_dir : str | Path
        Base directory containing Milo results (expects `<milo_dir>/<annotation>/`).
    exp_df : str | Path | pd.DataFrame
        Path to CSV of experiment metadata or a pre-loaded DataFrame. Must include:
        ['id', 'compare_variable', 'tissue_context', 'group_1', 'group_2'] and any columns referenced by `add_filter`.
    group_of_interest : str, optional
        Group for which positive logFC should indicate enrichment. See flipping logic above.
    only_show_group_of_interest : bool, optional
        Only keep experiments that include the group_of_interest.
    compartment : str, default "CD4"
        Which results file prefix to read (part of filename).
    savedir : str | Path, optional
        If provided, save the figure to this path (PNG/PDF based on extension).
    significant_only : bool, default True
        Keep only neighbourhoods with SpatialFDR < `sig_thresh` before plotting.
    sig_thresh : float, default 0.20
        Loose FDR threshold. Also used for "°" overlay when `sig_star=True`.
    sig_thresh_strict : float, default 0.10
        Strict FDR threshold for "*" overlay when `sig_star=True`.
    sig_star : bool, default False
        If True, overlay significance symbols (“°”, “*”) on the heatmap cells after clustering.
    blood_only : bool, default False
        If True, include only experiments where `tissue_context == "PBMCs"`.
    ignore_groups : sequence of str, optional
        Any experiment with group_1 or group_2 in this list will be skipped.
    add_filter : (str, str), optional
        A key/value exact-match filter applied to the experiment metadata (e.g., ("pathogen", "EBV")).
    drop_duplicates : bool, default False
        If True, drop duplicate experiments by 'subset_condition' (BUGFIX: uses subset='subset_condition').
    df_filter_exclude_annotation : sequence of str, default ("Mixed",)
        Annotations to exclude from the heatmap (e.g., remove noisy "Mixed").
    tissue_colors : dict, optional
        Mapping from tissue_context to color. Default: {"PBMCs": "firebrick", "non-PBMCs": "#999999"}.
    group_palette : sequence, optional
        Palette to use for group colors (defaults to seaborn tab20 of length 20).
    cmap : str, default "bwr"
        Heatmap colormap.
    center : float, default 0.0
        Value at which to center the diverging colormap.
    col_cluster : bool, default True
        Cluster columns in the clustermap.
    row_cluster : bool, default True
        Cluster rows in the clustermap.
    figure_size : (int, int), optional
        Figure size.
    return_objects : bool, default True
        If True, return (ClusterGrid, matrix_df, meta_df); otherwise return None.

    Returns
    -------
    (sns.matrix.ClusterGrid, pd.DataFrame, pd.DataFrame) or None
        The seaborn ClusterGrid, the plotted logFC matrix, and the per-experiment meta used to build row colors.

    Notes
    -----
    Expects each Milo CSV to contain at least:
        ['nhood_annotation', 'logFC', 'SpatialFDR'].
    """
    logging.getLogger(__name__).setLevel(logging.INFO)

    milo_dir = Path(milo_dir)
    summary_dir = milo_dir / annotation

    if isinstance(exp_df, (str, Path)):
        experiment_df = pd.read_csv(exp_df)
    else:
        experiment_df = exp_df.copy()

    # Basic input checks
    required_cols = {"id", "compare_variable", "tissue_context", "group_1", "group_2"}
    missing = required_cols - set(experiment_df.columns)
    if missing:
        raise ValueError(f"exp_df is missing required columns: {sorted(missing)}")

    experiment_df = experiment_df[experiment_df["compare_variable"] == compare_variable].copy()

    # BUGFIX: drop_duplicates previously used an invalid signature
    if drop_duplicates and "subset_condition" in experiment_df.columns:
        experiment_df = experiment_df.drop_duplicates(subset="subset_condition")

    if blood_only:
        experiment_df = experiment_df[experiment_df["tissue_context"] == "PBMCs"]

    if add_filter is not None:
        key, val = add_filter
        if key not in experiment_df.columns:
            raise KeyError(f"add_filter key '{key}' not found in exp_df columns")
        experiment_df = experiment_df[experiment_df[key] == val]

    if only_show_group_of_interest:
        experiment_df = experiment_df[
            experiment_df["group_1"].eq(group_of_interest) | experiment_df["group_2"].eq(group_of_interest)
        ]

    if ignore_groups:
        ig = set(ignore_groups)
        experiment_df = experiment_df[
            ~experiment_df["group_1"].isin(ig) & ~experiment_df["group_2"].isin(ig)
        ]

    all_results = []
    all_results_full = []
    selected_meta_rows = []

#     if tissue_colors is None:
#         tissue_colors = {"PBMCs": "firebrick", "non-PBMCs": "#999999"}

    for _, row in experiment_df.iterrows():
        eid = row["id"]
        summary_path = summary_dir / f"results_{compartment}_{eid}.csv"
        if not summary_path.exists():
            logging.warning(f"Missing file: {summary_path.name}")
            continue

        df = pd.read_csv(summary_path)
        missing_cols = {"nhood_annotation", "logFC", "SpatialFDR"} - set(df.columns)
        if missing_cols:
            logging.warning(f"{summary_path.name} missing columns: {sorted(missing_cols)}; skipping.")
            continue

        df["experiment_id"] = eid

        # Flip logFC to ensure consistent interpretation for group_of_interest.
        flipped = False
        if group_of_interest is not None:
            if group_of_interest == "Healthy":
                if row["group_1"] == group_of_interest:
                    df["logFC"] = -df["logFC"]
                    flipped = True
            else:
                if row["group_2"] == group_of_interest:
                    df["logFC"] = -df["logFC"]
                    flipped = True

        meta_row = row.copy()
        meta_row["id"] = eid
        if flipped:
            meta_row["group_1"], meta_row["group_2"] = meta_row["group_2"], meta_row["group_1"]
        selected_meta_rows.append(meta_row)

        if significant_only:
            df = df[df["SpatialFDR"] < sig_thresh].copy()

        if df_filter_exclude_annotation:
            df = df[~df["nhood_annotation"].isin(df_filter_exclude_annotation)]

        if df.empty:
            continue

        all_results.append(df[["experiment_id", "nhood_annotation", "logFC"]])
        if sig_star:
            all_results_full.append(df[["experiment_id", "nhood_annotation", "logFC", "SpatialFDR"]])

    if not all_results:
        logging.warning("No results to plot after filtering; nothing to do.")
        return None if not return_objects else (None, pd.DataFrame(), pd.DataFrame())

    # Build matrix and meta
    results_df = pd.concat(all_results, ignore_index=True)
    matrix = (
        results_df.pivot(index="experiment_id", columns="nhood_annotation", values="logFC")
        .fillna(0.0)
    )
    # Drop rows that are all zeros (no retained neighborhoods for that experiment)
    matrix = matrix.loc[~(matrix == 0).all(axis=1)]
    if matrix.empty:
        logging.warning("LogFC matrix is empty after zero-row removal; nothing to plot.")
        return None if not return_objects else (None, matrix, pd.DataFrame())

    meta = pd.DataFrame(selected_meta_rows).set_index("id").loc[matrix.index].copy()

    # --- Color mapping for groups ---
    # Stable ordering: put Healthy first if present, then alphabetical
    group_vals = pd.unique(meta[["group_1", "group_2"]].values.ravel("K"))
    group_vals = [g for g in group_vals if pd.notna(g)]
    if "Healthy" in group_vals:
        group_vals = ["Healthy"] + sorted([g for g in group_vals if g != "Healthy"])
    else:
        group_vals = sorted(group_vals)

    if group_palette is None:
        group_palette = sns.color_palette("tab20", n_colors=max(len(group_vals), 3))
    group_colors = dict(zip(group_vals, group_palette))

    row_colors = pd.DataFrame(
        {
            "Tissue": meta["tissue_context"].map(tissue_colors),
#             "Group 1": meta["group_1"].map(group_colors),
#             "Group 2": meta["group_2"].map(group_colors),
            "Group 1": meta["group_1"].map(pathogen_colors),
            "Group 2": meta["group_2"].map(pathogen_colors),
        },
        index=matrix.index,
    )
    if figure_size is not None:
        figsize = figure_size
    else:
        if matrix.shape[0] < 5: 
            figsize = (7, 6)
        else:
            figsize = (10, 10)

    g = sns.clustermap(
        matrix,
        row_colors=row_colors,
        cmap=cmap,
        center=center,
        figsize=figsize,
        xticklabels=True,
        yticklabels=False,
        col_cluster=col_cluster,
        row_cluster=row_cluster,
        cbar_kws={'label':'logFC'},
        cbar_pos=(1.12, 0.2, 0.03, 0.3),
        dendrogram_ratio=0.1,

    )

    # Optional significance overlay
    if sig_star and all_results_full:
        full_df = pd.concat(all_results_full, ignore_index=True)

        full_matrix = (
            full_df.pivot(index="experiment_id", columns="nhood_annotation", values="logFC")
            .reindex(index=matrix.index, columns=matrix.columns)
        )
        fdr = full_df.pivot(index="experiment_id", columns="nhood_annotation", values="SpatialFDR")
        fdr = fdr.reindex_like(full_matrix)

        sig_mask_loose = fdr < sig_thresh
        sig_mask_strict = fdr < sig_thresh_strict

        # Reordered indices from the clustermap
        row_order = [full_matrix.index[i] for i in g.dendrogram_row.reordered_ind]
        col_order = [full_matrix.columns[i] for i in g.dendrogram_col.reordered_ind]

        ax = g.ax_heatmap
        for i, rid in enumerate(row_order):
            for j, cid in enumerate(col_order):
                if pd.notna(sig_mask_loose.loc[rid, cid]) and sig_mask_loose.loc[rid, cid]:
                    mark = "*" if (pd.notna(sig_mask_strict.loc[rid, cid]) and sig_mask_strict.loc[rid, cid]) else "°"
                    ax.text(j + 0.5, i + 0.5, mark, ha="center", va="center", color="black", fontsize=12)

    # Legends
    tissue_legend = [Patch(facecolor=col, label=lbl) for lbl, col in (tissue_colors or {}).items() if lbl in meta.tissue_context.values]
#     group_legend = [Patch(facecolor=col, label=lbl) for lbl, col in group_colors.items()]
    group_legend = [Patch(facecolor=col, label=lbl) for lbl, col in pathogen_colors.items() if (lbl in meta.group_1.values) or (lbl in meta.group_2.values)]
    handles = tissue_legend + group_legend
    if handles:
        g.ax_heatmap.legend(
            handles=handles,
            loc="upper left",
            borderaxespad=0.0,
            title="Row Annotations",
            frameon=True,
            bbox_to_anchor=(1.2,1.3),
        )

    # Show experiment IDs on y-axis in clustered order
    reordered_index = matrix.index[g.dendrogram_row.reordered_ind]
    g.ax_heatmap.set_yticks(np.arange(len(reordered_index)) + 0.5)
    g.ax_heatmap.set_yticklabels(reordered_index)

    plt.suptitle(f"FDR: {sig_thresh} > ° > {sig_thresh_strict} > *, {annotation}", y=1.05,)
    plt.tight_layout()

    if savedir is not None:
        savedir = Path(savedir)
        g.savefig(savedir, bbox_inches="tight")

    plt.show()

    return (g, matrix, meta) if return_objects else None

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 227
# for compartment in ['CD4']:
for annotation in ['annotation_level_2','annotation_level_3','annotation_level_4']:
    for compartment in ['CD8','CD4']:
        plot_multi_milo(
            compare_variable='pathogen', 
            group_of_interest='Healthy',
            only_show_group_of_interest=True,
            annotation=annotation, 
            compartment=compartment,
            exp_df=root_dir+'datasets/pan_infection_atlas/miscellaneous/milo_checkpoint_6/milo_experiment_plan_7.csv',
            milo_dir=root_dir+'datasets/pan_infection_atlas/miscellaneous/milo_checkpoint_6/',
            savedir=savedir+f'milo/milo_{compartment}_{annotation}_multiFDR',#_including_nonsignificant',
            significant_only=False,
            sig_star=True,
            sig_thresh=0.2,
            sig_thresh_strict=0.1,
            figure_size=(10,8),
    #             blood_only=True,
            ignore_groups=['EBV_VZV','Influenza_CMV_EBV'],
    #         add_filter=('disease_stage','Convalescent'),
            drop_duplicates=True,
        )

In [ ]:
# Source: pan_infection_atlas_figures.ipynb cell 228
# for compartment in ['CD4']:
for annotation in ['annotation_level_2','annotation_level_3','annotation_level_4']:
    for compartment in ['CD8']:
        plot_multi_milo(
            compare_variable='pathogen', 
            group_of_interest='Healthy',
            only_show_group_of_interest=True,
            annotation=annotation, 
            compartment=compartment,
            exp_df='/nfs/team205/tcr_pipeline/datasets/pan_infection_atlas/miscellaneous/milo_checkpoint_6/milo_experiment_plan_7.csv',
            milo_dir='/nfs/team205/tcr_pipeline/datasets/pan_infection_atlas/miscellaneous/milo_checkpoint_6/',
            savedir=savedir+f'milo_{compartment}_{annotation}_multiFDR',#_including_nonsignificant',
            significant_only=False,
            sig_star=True,
            sig_thresh=0.2,
            sig_thresh_strict=0.1,
            figure_size=(10,8),
    #             blood_only=True,
            ignore_groups=['EBV_VZV','Influenza_CMV_EBV'],
    #         add_filter=('disease_stage','Convalescent'),
            drop_duplicates=True,
        )

## 5. Load TCR motif table for downstream specificity analysis

This starts the later TCR motif analysis from the second notebook.

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 35
import pandas as pd
import cell2tcr
# Cell2TCR motifs
df = pd.read_csv('/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection_cell2tcr/cell2tcr_job/clone_df_with_motif_21.csv', index_col='Unnamed: 0')
df.head()

# attach BEAM specificities
root_dir = '/home/lmd76/rfs/rfs-teichlab-nfs-iCNyzSAaucw/lmd76/pan_infection/'
beam_labels = pd.read_csv(root_dir+'datasets/pan_infection_atlas/miscellaneous/milo_proper/BEAM_pathogen_split_detail.csv', index_col=1)
beam_labels[['beam_hla','beam_epitope','beam_organism']] = beam_labels.virus_detail.str.split('_', expand=True).iloc[:,:3].values
beam_labels


beam_labels = beam_labels[beam_labels.index.isin(df.index)]
df.loc[beam_labels.index, ['beam_hla','beam_epitope','beam_organism']] = beam_labels[['beam_hla','beam_epitope','beam_organism']].values


import numpy as np
# split BEAM VZV-EBV cells 
df.loc[df.study=='GSE275633',['beam_organism','beam_epitope']] = df.loc[df.study=='GSE275633', 'antigen'].str.split('_', expand=True).values
df.loc[df.study=='GSE275633','pathogen'] = df.loc[df.study=='GSE275633', 'antigen'].apply(lambda x: 'EBV_VZV' if isinstance(x, float) else 'EBV' if 'EBV' in x else 'VZV' if 'VZV' in x else np.nan)

## 6. Motif presence matrices for HLA and pathogen inference

Generate HLA/pathogen motif presence plots and cross-validation performance figures.

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 54
motifs_disease = pd.read_csv('data/disease_associated_motifs_hla.csv')
motifs_disease

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 55
df_motifs_hla = pd.read_csv('data/df_motifs_with_hla.csv')
df_motifs_hla

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 56
sel_motifs = df_motifs_hla[(df_motifs_hla.MHC_I_restricted)&(df_motifs_hla.is_CD8>0.5)&(df_motifs_hla.n_donors>=4)&(df_motifs_hla.MHC_I_restricted_allele!='Multiple')].motif.values

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 57
import seaborn as sns

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 58
motif_hla_map = df_motifs_hla.set_index('motif').MHC_I_restricted_allele.to_dict()

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 59
n = 30
data = pd.crosstab(
    df[df.motif.isin(sel_motifs[:n])].donor_id,
    df[df.motif.isin(sel_motifs[:n])].motif,
#     normalize=True
).gt(0)

hla_series = data.columns.map(motif_hla_map)
data = data.iloc[:, hla_series.argsort()]

lut = dict(zip(sorted(data.columns.map(motif_hla_map).unique()), sns.color_palette("tab20", 25)))
col_colors = data.columns.map(motif_hla_map).to_series().map(lut)

g = sns.clustermap(
    data=data, dendrogram_ratio=0.1, figsize=(8,8),
    col_colors=col_colors.values, cmap='Blues',
    cbar_pos=None,
    col_cluster=False,
)
plt.suptitle(
    f'Presence of {n} most common HLA-restricted CD8 TCR motifs across atlas\n(detected in n={data.shape[0]} donors)',
    y=0.97,
)
g.ax_heatmap.set_yticks([])

import matplotlib.patches as mpatches

handles = [mpatches.Patch(color=c, label=l) for l, c in lut.items()]
g.fig.legend(handles=handles, title='HLA Allele', loc='upper right', bbox_to_anchor=(0, 0.6))
plt.savefig(savedir+'presence_30_hla_tcrmotifs.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 60
sel_motifs_hla = df_motifs_hla[
    (df_motifs_hla.MHC_I_restricted)&
    (df_motifs_hla.is_CD8>0.5)&
    (df_motifs_hla.n_donors>=4)&
    (df_motifs_hla.MHC_I_restricted_allele!='Multiple')].motif.values

sel_motifs_pathogen = motifs_disease.set_index('motif').predicted_pathogen.dropna().index.values

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 61
pathogen_colors = {
    "CMV": "#393b79",
    "Dengue_virus": "#5254a3",
    "EBV": "#6b6ecf",
    "HBV": "#637939",
    "HCV": "#8ca252",
    "HIV": "#b5cf6b",
    "HPV": "#8c6d31",
#     "HSV-2": "#bd9e39",
    "Yellow fever": "#bd9e39",
    "Healthy": "#e7ba52",
    "Influenza_CMV_EBV": "#843c39",
    "Influenza virus": "#ad494a",
    "Influenza": "#ad494a",
    "Plasmodium_falciparum": "#d6616b",
    "SARS-CoV-2": "#7b4173",
    "SARS-CoV-1": "tomato",
    "Tuberculosis": "#a55194",
    "VZV": "#ce6dbd",
    "other": "#de9ed6",
}
# pathogen_colors

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 62
n_patho = 3
patho_map = (motifs_disease.sort_values(['predicted_pathogen','motif'])
             .groupby('predicted_pathogen')
             .head(n_patho)[['predicted_pathogen', 'motif']]).reset_index(drop=True)
patho_map = patho_map[patho_map.predicted_pathogen.isin(['CMV', 'EBV', 'HIV', 'HPV', 'Influenza','SARS-CoV-2', 'VZV',])]
# patho_map

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 63
data = pd.crosstab(
    df[df.motif.isin(patho_map.motif.unique())].donor_id,
    df[df.motif.isin(patho_map.motif.unique())].motif,
).gt(0)

pathogen_series = data.columns.map(patho_map.set_index('motif').predicted_pathogen.to_dict())

# lut = dict(zip(sorted(patho_map.predicted_pathogen.unique()), sns.color_palette("tab20", 25)))
lut = pathogen_colors
data = data.reindex(columns=patho_map.motif)

col_colors = patho_map.predicted_pathogen.map(lut)

g = sns.clustermap(
    data=data, dendrogram_ratio=0.1, figsize=(6,7),
    col_colors=col_colors.values, cmap='Blues',
    cbar_pos=None,
    col_cluster=False,
)
plt.suptitle(
    f'Presence of {n_patho} most common pathogen-restricted TCR motifs\nper pathogen across atlas (detected in n={data.shape[0]} donors)',
    y=0.97,
)
g.ax_heatmap.set_yticks([])

import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=c, label=l) for l, c in lut.items() if l in patho_map.predicted_pathogen.unique()]
g.fig.legend(handles=handles, title='Pathogen', loc='upper right', bbox_to_anchor=(0, 0.6))
plt.savefig(savedir+'presence_3_pathogen_tcrmotifs.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 65
motif_to_patho = motifs_disease.set_index('motif').predicted_pathogen.dropna().to_dict()
motif_to_hla = df_motifs_hla[
    (df_motifs_hla.MHC_I_restricted)&
    (df_motifs_hla.is_CD8>0.5)&
    (df_motifs_hla.n_donors>=4)&
    (df_motifs_hla.MHC_I_restricted_allele!='Multiple')].set_index('motif').MHC_I_restricted_allele.to_dict()

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 66
donor_motif_matrix = pd.crosstab(df.donor_id, df.motif).gt(0).astype(int)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 67
pathogen_scores = donor_motif_matrix.groupby(motif_to_patho, axis=1).sum()
hla_scores = donor_motif_matrix.groupby(motif_to_hla, axis=1).sum()
donor_features = pd.concat([pathogen_scores, hla_scores], axis=1)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 70
# ground truth: arcas-hla inferred HLA types
hla = pd.read_csv(root_dir+'datasets/pan_infection_atlas/miscellaneous/hla_combat_ren_lindeboom_eskd_beam.csv', index_col=0)
hla = hla[hla.index.isin(df.donor_id)]
hla_arcas = hla.iloc[:,:-1]
hla.shape

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 71
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve
)

# ----------------------------
# 1. Align donors and HLA columns
# ----------------------------
common_donors = hla_arcas.index.intersection(hla_scores.index)
common_hla = hla_arcas.columns.intersection(hla_scores.columns)

X = hla_scores.loc[common_donors, common_hla].copy()
Y = hla_arcas.loc[common_donors, common_hla].copy()

# ----------------------------
# 2. Helper functions
# ----------------------------
def choose_threshold_from_train(y_true, score):
    """
    Pick threshold on training data using Youden's J statistic.
    Ignores non-finite thresholds.
    """
    fpr, tpr, thresholds = roc_curve(y_true, score)
    thresholds = np.asarray(thresholds)
    j = tpr - fpr

    finite = np.isfinite(thresholds)
    if finite.sum() == 0:
        return 1.0

    best_idx = np.argmax(j[finite])
    return thresholds[finite][best_idx]

def eval_on_test(y_true, score, threshold):
    y_pred = (score >= threshold).astype(int)
    return {
        "Test_AUC": roc_auc_score(y_true, score) if y_true.nunique() > 1 else np.nan,
        "Test_BalAcc": balanced_accuracy_score(y_true, y_pred),
        "Test_F1": f1_score(y_true, y_pred, zero_division=0),
        "Test_Precision": precision_score(y_true, y_pred, zero_division=0),
        "Test_Recall": recall_score(y_true, y_pred, zero_division=0),
        "Threshold": threshold,
        "N_test": len(y_true),
        "Pos_test": int(y_true.sum())
    }

# ----------------------------
# 3. Per-HLA train/test evaluation
# ----------------------------
results = []

for hla in common_hla:
    y = Y[hla]

    # skip HLAs without both classes
    if y.nunique() < 2:
        continue

    try:
        train_idx, test_idx = train_test_split(
            y.index,
            test_size=0.2,
            random_state=42,
            stratify=y
        )
    except ValueError:
        # skip very rare alleles that cannot be split properly
        continue

    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]

    # need both classes in training for threshold selection
    if y_train.nunique() < 2 or y_test.nunique() < 2:
        continue

    s_train = X.loc[train_idx, hla]
    s_test = X.loc[test_idx, hla]

    threshold = choose_threshold_from_train(y_train, s_train)
    metrics = eval_on_test(y_test, s_test, threshold)

    results.append({
        "HLA": hla,
        **metrics
    })

results_df = pd.DataFrame(results).set_index("HLA").sort_values("Test_AUC", ascending=False)
results_df

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 74
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_curve

# ----------------------------
# 1. Setup CV and results container
# ----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
final_results = []
full = []
for hla in common_hla:
    y = Y[hla]
    
    # Requirement: at least 5 positive samples to do 5-fold stratified CV
    if y.sum() < 5 or (len(y) - y.sum()) < 5:
        continue

    fold_metrics = []
    
    # ----------------------------
    # 2. Loop through Folds
    # ----------------------------
    # Using .values for index safety with StratifiedKFold
    indices = np.arange(len(y))
    
    for train_idx_raw, test_idx_raw in skf.split(indices, y):
        # Map back to original dataframe indices
        train_idx = y.index[train_idx_raw]
        test_idx = y.index[test_idx_raw]
        
        y_train, y_test = y.loc[train_idx], y.loc[test_idx]
        s_train, s_test = X.loc[train_idx, hla], X.loc[test_idx, hla]
        
        # Train threshold on this fold's training data
        threshold = choose_threshold_from_train(y_train, s_train)
        
        # Evaluate on this fold's test data
        metrics = eval_on_test(y_test, s_test, threshold)
        metrics['threshold'] = threshold
        fold_metrics.append(metrics)
    
    # ----------------------------
    # 3. Aggregate across Folds
    # ----------------------------
    df_folds = pd.DataFrame(fold_metrics)
    
    final_results.append({
        "HLA": hla,
        "Mean_AUC": df_folds["Test_AUC"].mean(),
        "Std_AUC": df_folds["Test_AUC"].std(),
        "Mean_BalAcc": df_folds["Test_BalAcc"].mean(),
        "Mean_F1": df_folds["Test_F1"].mean(),
        "Mean_Precision": df_folds["Test_Precision"].mean(),
        "Mean_Recall": df_folds["Test_Recall"].mean(),
        "N_total": len(y),
        "Pos_total": int(y.sum())
    })
    df_folds['HLA'] = hla
    df_folds['Pos_total'] = int(y.sum())
    full.append(df_folds)

# Final DataFrame for the paper
results_df = pd.DataFrame(final_results).set_index("HLA").sort_values("Mean_AUC", ascending=False)
results_df

In [ ]:
from collections import defaultdict

hla_to_motifs = defaultdict(list)
for motif, allele in motif_to_hla.items():
    if motif in donor_motif_matrix.columns:
        hla_to_motifs[allele].append(motif)

hla_motif_counts = pd.DataFrame(hla_to_motifs.items(), columns=["HLA", "motifs"])
hla_motif_counts["n_motifs"] = hla_motif_counts["motifs"].apply(len)

data = pd.concat(full)
data["n_motifs"] = data["HLA"].map(hla_motif_counts.set_index("HLA")["n_motifs"].to_dict())
data.head()


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 78
sns.barplot(
    data=data,
    x='HLA',
    y='Test_AUC',
    hue='n_motifs',
    order=results_df.index,
#     palette='viridis',
)
plt.xticks(rotation=90)
plt.axhline(0.5, color='k', linestyle='--', alpha=0.6, label='Random')
plt.legend(bbox_to_anchor=(1.1,.6), title='Number of motifs')
plt.ylim(0,1)
plt.title('Performance of HLA prediction via 5-fold stratified cross-validation')
plt.savefig(savedir+'predicted_hla_auc_5cv.pdf')
plt.show()

In [ ]:
# Predict HLA calls for donors without ArcasHLA ground truth using motif-count evidence.
# This replaces an exploratory notebook cell that depended on an in-memory `trained_models` object.
all_donors = donor_motif_matrix.index
known_donors = hla_arcas.index
held_out_donors = all_donors.difference(known_donors)

prob_df = hla_scores.loc[held_out_donors].copy()
binary_df = prob_df.gt(0).astype(int)

print(f"Total positive predictions across all HLAs: {binary_df.sum().sum()}")


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 81
g = sns.clustermap(
    binary_df.loc[binary_df.sum(axis=1).gt(0),binary_df.sum(axis=0).gt(0)], 
    cmap='Blues', cbar_pos=None, col_cluster=False, figsize=(8,8),
    dendrogram_ratio=0.1,
)
plt.suptitle(
    f'Predicted HLA types\n(available for {binary_df[binary_df.sum(axis=1).gt(0)].shape[0]} out of {binary_df.shape[0]} donors with unknown HLA type)',
    y=1.,
)
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_ylabel('donor_id')
plt.savefig(savedir+'predicted_hla_clustermap.pdf')
plt.show()

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 82
# 1. Ensure we have the boolean predictions (using your chosen threshold)
# predicted_types is a Donor x HLA boolean matrix
threshold = 0.5
predicted_types = prob_df > threshold

# 2. Group the columns by Locus (A, B, C)
# This assumes your columns are named like 'A*02:01', 'B*07:02', etc.
a_cols = [c for c in predicted_types.columns if c.startswith('A*')]
b_cols = [c for c in predicted_types.columns if c.startswith('B*')]
c_cols = [c for c in predicted_types.columns if c.startswith('C*')]

# 3. Calculate counts per donor
counts = pd.DataFrame({
    'A': predicted_types[a_cols].sum(axis=1),
    'B': predicted_types[b_cols].sum(axis=1),
    'C': predicted_types[c_cols].sum(axis=1)
})

# 4. Reshape for plotting (Melt)
counts_melted = counts.melt(var_name='Locus', value_name='Predicted_Alleles').query('Predicted_Alleles>0')

# 5. Create the Visualization
plt.figure(figsize=(4,3))
sns.countplot(data=counts_melted, x='Predicted_Alleles', hue='Locus', palette='bone')

# Add the "Sanity Line" at 2
# plt.axvline(x=2.5, color='red', linestyle='--', alpha=0.6, label='Biological Limit')

plt.title('Predicted alleles per locus')
plt.xlabel('Number of Unique Alleles Predicted')
plt.ylabel('Number of Donors')
plt.legend(title='HLA Locus')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(savedir+'predicted_alleles_per_locus.pdf')
plt.show()

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 84
from collections import defaultdict
pathogen_to_motifs = defaultdict(list)
for motif, pathogen in motif_to_patho.items():
    if motif in donor_motif_matrix.columns:
        pathogen_to_motifs[pathogen].append(motif)

pathogen_scores = pd.DataFrame(index=donor_motif_matrix.index)

for pathogen, motifs in pathogen_to_motifs.items():
    pathogen_scores[pathogen] = donor_motif_matrix[motifs].sum(axis=1)

pathogen_scores

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 87
from scipy.stats import chi2_contingency

true_labels = pd.crosstab(df.donor_id, df.pathogen)['SARS-CoV-2'].fillna(0).gt(0)
pred_labels = pathogen_scores['SARS-CoV-2'].gt(1)

true_labels.index = true_labels.index.astype(str)
pred_labels.index = pred_labels.index.astype(str)

true_labels_clean = true_labels.groupby(level=0).max()
pred_labels_clean = pred_labels.groupby(level=0).max()

# 3. Align and drop donors that don't exist in both datasets
comparison_df = pd.DataFrame({
    'True': true_labels_clean,
    'Predicted': pred_labels_clean
}).dropna()

# 4. Create the 2x2 Contingency Table
# (Rows = Clinical Truth, Columns = TCR Prediction)
contingency_table = pd.crosstab(comparison_df['True'], comparison_df['Predicted'])

# 5. Run the Test
chi2, p, dof, expected = chi2_contingency(contingency_table)


plt.figure(figsize=(2,2))
sns.heatmap(contingency_table, cmap='Blues', annot=True, fmt='.4g')
plt.xlabel('Predicted COVID-19 status')
plt.ylabel('True COVID-19 status')
plt.title(f"threshold > 1\nChi-squared Statistic: {chi2:.2f}\nP-value: {p:.3e}")
plt.savefig(savedir+'exposure_prediction_sarscov2_chi2.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 89
patho_order = ['CMV','EBV', 'HPV', 'Influenza', 'SARS-CoV-2', 'VZV']
data = pathogen_scores.gt(1)
data = data.loc[:, patho_order]

g = sns.clustermap(
    data=data[data.sum(axis=1).gt(0)].T,
    cmap='Blues',
    cbar_pos=None,
    figsize=(8,5),
    row_cluster=False,
    dendrogram_ratio=0.1,
)
plt.suptitle(
    f'Predicted exposure status for {data.sum(axis=1).gt(0).sum()} out of {data.shape[0]} donors\n(evidence of 2 or more pathogen-associated motifs)',
    y=1.1,
)
g.ax_heatmap.set_xticks(ticks=[])
g.ax_heatmap.yaxis.tick_left()

plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)

plt.savefig(savedir+'exposed_donor_clustermap.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 90
plt.figure(figsize=(2,4))
pathogen_scores.gt(1).loc[:,reversed(patho_order)].sum().plot.barh(
    title='Predicted exposed donors', logx=True, xlabel='Count', cmap='Blues_r')
plt.savefig(savedir+'exposed_donor_count.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 92
import pandas as pd
import matplotlib.pyplot as plt

# Assumes df has columns: donor_id, clone_id, annotation_level_2
# Each row is a cell

# 1) Clone size within donor (per cell)
# df = df.copy()
df["clone_size"] = df.groupby(["donor_id", "clone_id"])["clone_id"].transform("size")

# 2) Bin expansion per cell
df["exp_bin"] = pd.cut(
    df["clone_size"],
    bins=[0, 1, 9, float("inf")],
    labels=["1x", "2-9x", "10x+"],
    right=True
)

# 3) Per state: cell counts and proportions by expansion bin
cell_counts = (
    df.groupby(["annotation_level_3", "exp_bin"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=["1x", "2-9x", "10x+"], fill_value=0)
)

cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)

# 4) Optional: per state, unique clone counts and proportions by expansion bin
clone_counts = (
    df.drop_duplicates(["donor_id", "clone_id", "annotation_level_3"])
      .groupby(["annotation_level_3", "exp_bin"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=["1x", "2-9x", "10x+"], fill_value=0)
)

clone_props = clone_counts.div(clone_counts.sum(axis=1), axis=0)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 93
custom_order_CD8 = [
    'T CD8 Naive RTE',
    'T CD8 Naive',
    'T CD8 CM',
    'T CD8 EM',
    'T CD8 Activated',
    'T CD8 EF',
    'T CD8 EF KIR',
    'T CD8 RM',
    'T CD8 SLEX',
    'T CD8 PEX',
    'T CD8 EX',
    'T CD8 Reg'
]

custom_order_CD4 = [
 'T CD4 Naive RTE',
 'T CD4 Naive',
 'T CD4 CM TH0',
 'T CD4 EM TH0 GZMK',
 'T CD4 EM TH0',
 'T CD4 EF TH0',    
 'T CD4 CM TH1',
 'T CD4 EM TH1',
 'T CD4 EM TH1 GZMK',
 'T CD4 EF TH1',    
 'T CD4 CM TH2',
 'T CD4 EM TH2',    
 'T CD4 EF TH2',
 'T CD4 CM TH17',
 'T CD4 EM TH17',
 'T CD4 EM TH17 GZMK',
 'T CD4 EF TH17',
 'T CD4 CM TH1/17',
 'T CD4 EM TH1/17',
 'T CD4 EM TH22',
 'T CD4 Activated TH1',
 'T CD4 Activated CTL',
 'T CD4 CTL',
 'T CD4 FH',
 'T CD4 PH',
 'T CD4 RM',
 'T CD4 Reg Naive',
 'T CD4 Reg Memory',
 'T CD4 Reg Activated'
]

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 94
# 5) Plot: stacked bar of per-state proportions (cells)
ax = cell_props.loc[reversed(custom_order_CD8)].plot(kind="barh", stacked=True, figsize=(5,6), color=["silver", "darkorange", "firebrick"])
ax.set_xlabel("Fraction of cells")
ax.set_ylabel("Cell state (annotation_level_3)")
# ax.set_title("Clonal expansion bins per cell state (average within-donor clone sizes)")
ax.legend(title="Within-donor\nclonal expansion", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(savedir+'clonal_expansion_annotation_level_3_CD8')
plt.show()

ax = cell_props.loc[reversed(custom_order_CD4)].plot(kind="barh", stacked=True, figsize=(5,6), color=["silver", "darkorange", "firebrick"])
ax.set_xlabel("Fraction of cells")
ax.set_ylabel("Cell state (annotation_level_3)")
# ax.set_title("Clonal expansion bins per cell state (average within-donor clone sizes)")
ax.legend(title="Within-donor\nclonal expansion", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(savedir+'clonal_expansion_annotation_level_3_CD4')
plt.show()

## 7. Disease-associated TCR motif annotation and motif summary figures

Create the disease-associated motif reference table and generate saved motif summary plots.

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 107
def strip_allele(gene: str) -> str:
    if gene is None:
        return ""
    s = str(gene)
    if s == "nan":
        return ""
    return s.split("*", 1)[0]

def summarize_motifs_majority_labels(
    df: pd.DataFrame,
    motif_col: str = "motif",
    pathogen_col: str = "pathogen",
    annot_col: str = "annotation_level_1",
    naive_annot_col: str = "annotation_level_2",
    donor_col: str = "donor_id",
    study_col: str = "study",
    vdj_aa_col: str = "vdj_aa",
    vdj_v_col: str = "vdj_v",
    vdj_j_col: str = "vdj_j",
    healthy_label: str = "Healthy",
) -> pd.DataFrame:
    """
    For each motif:
      - Majority labels for pathogen and annotation_level_1 are based on cell counts (rows).
      - Confidence metrics include:
          * majority cell fraction
          * proportion of unique VDJ clones (V, J, CDR3aa) supporting that majority label
      - Also reports:
          * n_cells, n_unique_vdj_clones, n_donors, n_studies
          * proportion of unique VDJ clones with pathogen == healthy_label
    """

    d = df.copy()

    # Build a "unique VDJ clone" key: allele-stripped V and J + beta CDR3 AA
    d["_vdj_v_stripped"] = d[vdj_v_col].map(strip_allele)
    d["_vdj_j_stripped"] = d[vdj_j_col].map(strip_allele)
    d["_vdj_aa_str"] = d[vdj_aa_col].astype(str)

    # Treat empty / nan-like as missing for clone identity
    missing_clone = (
        d["_vdj_aa_str"].eq("nan") | d["_vdj_aa_str"].eq("") |
        d["_vdj_v_stripped"].eq("") | d["_vdj_j_stripped"].eq("")
    )
    d["_vdj_clone_key"] = (
        d["_vdj_v_stripped"].astype(str)
        + "|"
        + d["_vdj_j_stripped"].astype(str)
        + "|"
        + d["_vdj_aa_str"].astype(str)
    )
    d.loc[missing_clone, "_vdj_clone_key"] = pd.NA

    def _majority_and_clone_frac(g: pd.DataFrame, label_col: str):
        # Majority by cell counts
        # Ignore cells with the same clone_id (i.e. multiple cells from the same donor)
        g = g.drop_duplicates(subset=["clone_id"], keep="first")
        vc = g[label_col].astype("string").fillna(pd.NA).value_counts(dropna=False, normalize=True)
        if vc.empty:
            maj = pd.NA
            maj_cell_frac = 0.0
            maj_clone_frac = 0.0
            tie = False
        else:
            top = vc.iloc[0]
            maj = vc.index[0]
            maj_cell_frac = float(top)

            # Tie flag (useful as a confidence indicator)
            tie = (len(vc) > 1 and vc.iloc[1] == top)

            # Proportion of unique VDJ clones supporting the majority label
            total_clones = g["clone_id"].nunique(dropna=True) # problem: same clone can appear with multiple 'pathogen' labels (i.e. healthy)
            maj_clones = g.loc[g[label_col] == maj, "clone_id"].nunique(dropna=True)
            maj_clone_frac = float(maj_clones / total_clones)

        return maj, maj_cell_frac, maj_clone_frac, tie

    rows = []
    for motif, g in d.groupby(motif_col, dropna=False):
        n_cells = int(len(g))
        n_unique_vdj_clones = int(g["_vdj_clone_key"].nunique(dropna=True))
        n_beam_cells = int(g['beam_epitope'].notna().sum()) if 'beam_epitope' in g.columns else 0
        n_donors = int(g[donor_col].nunique(dropna=False)) if donor_col in g.columns else 0
        n_studies = int(g[study_col].nunique(dropna=False)) if study_col in g.columns else 0

        maj_pathogen, maj_path_cell_frac, maj_path_clone_frac, path_tie = _majority_and_clone_frac(g, pathogen_col)
        
        # Proportion of unique VDJ clones from healthy
        healthy_donor_frac = g.drop_duplicates(['donor_id',pathogen_col])[pathogen_col].value_counts(normalize=True).get(healthy_label, 0.0)

        # as annotations were performed per-cell, don't drop duplicates
        vc_annot = g[annot_col].value_counts(normalize=True)
        if vc_annot.empty:
            maj_annot = pd.NA
            maj_ann_cell_frac = 0.0
        else:
            maj_annot, maj_ann_cell_frac = next(vc_annot.items())
        naive_ann_cell_frac = g[naive_annot_col].str.lower().str.contains('naive').sum() / n_cells if n_cells > 0 else 0.0

        rows.append({
            motif_col: motif,
            "n_cells": n_cells,
            "n_unique_vdj_clones": n_unique_vdj_clones,
            "n_beam_cells": n_beam_cells,
            "n_donors": n_donors,
            "n_studies": n_studies,

            "majority_pathogen": maj_pathogen,
#             "majority_pathogen_cell_frac": maj_path_cell_frac,
            "majority_pathogen_unique_clone_frac": maj_path_clone_frac,
#             "majority_pathogen_tied": path_tie,

            "majority_annotation": maj_annot,
#             "majority_annotation_level_1_cell_frac": maj_ann_cell_frac,
            "majority_annotation_all_cells_frac": maj_ann_cell_frac,
            "naive_annotation_all_cells_frac": naive_ann_cell_frac,
#             "majority_annotation_level_1_tied": ann_tie,

            "healthy_unique_donor_frac": healthy_donor_frac,
            'majority_vdjdb_epitope': g.drop_duplicates('_vdj_clone_key')['vdjdb_epitope'].mode().iloc[0] if not g.drop_duplicates('_vdj_clone_key')['vdjdb_epitope'].mode().empty else pd.NA,
            'majority_vdjdb_organism': g.drop_duplicates('_vdj_clone_key')['vdjdb_organism'].mode().iloc[0] if not g.drop_duplicates('_vdj_clone_key')['vdjdb_organism'].mode().empty else pd.NA,
            "n_vdjdb_epitopes": g['vdjdb_epitope'].nunique(dropna=True) if not g['vdjdb_epitope'].value_counts(dropna=True).empty else pd.NA,

            # don't drop duplicates here as beam labels are per-cell
            'majority_beam_epitope': g['beam_epitope'].mode(dropna=True).iloc[0] if not g['beam_epitope'].mode(dropna=True).empty else pd.NA, 
            'majority_beam_organism': g['beam_organism'].mode(dropna=True).iloc[0] if not g['beam_organism'].mode(dropna=True).empty else pd.NA,
            'majority_beam_cells_frac': g['beam_epitope'].value_counts(dropna=True, normalize=True).iloc[0] if not g['beam_epitope'].value_counts(dropna=True, normalize=True).empty else pd.NA,
        })

    out = pd.DataFrame(rows).sort_values(["n_cells", "n_unique_vdj_clones"], ascending=False).reset_index(drop=True)
    return out


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 108
# Filter to motifs with 4 or more donors (public motifs)
motifs_4plus_donors = df.groupby('motif').nunique().donor_id[df.groupby('motif').nunique().donor_id.gt(3)].index
df_sub = df[df.motif.isin(motifs_4plus_donors)]
# df_sub

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 112
db_match = pd.read_csv('db_match_motifs_4plus_donors.csv') 

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 113
# Rename columns
db_match = db_match.rename(columns={
    'hla': 'vdjdb_hla',
    'epitope': 'vdjdb_epitope',
    'organism': 'vdjdb_organism'
})
db_match = db_match.sort_values('score', ascending=False).drop_duplicates('input_sequence')

# Merge instead of map
df_sub = df_sub.merge(
    db_match.rename(columns={'input_sequence': 'vdj_aa'}),
    on='vdj_aa',
    how='left'
)


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 114
for col in ['vdjdb_epitope','vdjdb_organism']:
    df_sub[col] = df_sub[col].str.split(',', expand=True)[0]


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 119
df_m = summarize_motifs_majority_labels(df_sub, annot_col='annotation_level_1', naive_annot_col='annotation_level_2')
# df_m = summarize_motifs_majority_labels(df_sub, annot_col='annotation_level_2') # for all TCR motifs
df_m

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 125
# these are public TCR motifs with disease relevance
df_m_disease = df_m[
    (df_m.n_donors.ge(4))& # public motif - shared by 4 or more donors
    (df_m.healthy_unique_donor_frac.lt(0.2))& # absent in healthy controls (less than 20% of donors are healthy)
    ~(df_m.majority_annotation.fillna('').str.contains('MAI'))&  # not MAIT cell
    df_m.naive_annotation_all_cells_frac.lt(0.2)  # not naive T cell
    ]
df_m_disease

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 126
organism_map = {
    "Human herpesvirus 5 (Human cytomegalovirus)": "CMV",
    "Human herpesvirus 5 (strain RV798)": "CMV",
    "Human herpesvirus 5 strain AD169 (Human cytomegalovirus (strain AD169))": "CMV",
    "Human herpesvirus 5 strain Towne (Human cytomegalovirus (strain Towne))": "CMV",

    "Human herpesvirus 4 (Epstein Barr virus)": "EBV",
    "Human herpesvirus 4 strain B95-8 (Epstein-Barr virus (strain B95-8))": "EBV",

    "SARS-CoV2": "SARS-CoV-2",

    "SARS coronavirus Urbani (SARS-CoV (Urbani strain))": "SARS-CoV-1",
    "SARS coronavirus BJ01": "SARS-CoV-1",

    "Homo sapiens (human)": "other",

    "Influenza A virus": "Influenza",
    "Influenza A virus (A/Puerto Rico/8/1934(H1N1)) (Influenza A virus (A/PR 8/34 (H1N1)))": "Influenza",
    "Influenza A virus (A/Netherlands/602/2009(H1N1))": "Influenza",

    "Yellow fever virus 17D (Yellow fever virus (STRAIN 17D))": "Yellow fever",

    "Mycobacterium tuberculosis H37Rv (Mycobacterium tuberculosis str. H37Rv)": "Tuberculosis",
    "Mycobacterium tuberculosis": "Tuberculosis",

    "Murid herpesvirus 1 deltaMS94.5": "other",
    "AKR (endogenous) murine leukemia virus (AKR murine leukemia virus)": "other",

    "Triticum aestivum (Canadian hard winter wheat)": "other",
}

df_m_disease['majority_vdjdb_organism_simple'] = df_m_disease.majority_vdjdb_organism.map(organism_map)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 127
priority_pathogens = ['HPV', 'SARS-CoV-2', 'EBV', 'HIV', 'Dengue_virus']

# Mask for the extra rule (only used when BEAM is missing)
m_beam = df_m_disease['majority_beam_organism'].notna()
m_priority = (
    ~m_beam
    & df_m_disease['majority_pathogen_unique_clone_frac'].gt(0.7)
    & df_m_disease['majority_pathogen'].isin(priority_pathogens)
)

# Predicted pathogen: BEAM > priority rule > VDJdb > NA
df_m_disease['predicted_pathogen'] = np.select(
    [m_beam, m_priority, df_m_disease['majority_vdjdb_organism_simple'].notna()],
    [df_m_disease['majority_beam_organism'],
     df_m_disease['majority_pathogen'],
     df_m_disease['majority_vdjdb_organism_simple']],
    default=pd.NA
)

# Source flag
df_m_disease['predicted_pathogen_source'] = np.select(
    [m_beam, m_priority, df_m_disease['majority_vdjdb_organism_simple'].notna()],
    ['beam', 'majority_pathogen', 'vdjdb'],
    default=pd.NA
)

# Convert empty strings to NA
df_m_disease['predicted_pathogen'] = df_m_disease['predicted_pathogen'].replace({'': pd.NA})

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 129
df_hla = pd.read_csv('data/df_motifs_with_hla.csv')
# df_hla = df_hla[['motif','MHC_I_restricted_allele','MHC_II_restricted_allele']].set_index('motif')
df_hla

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 133
df_m_disease = df_m_disease.join(df_hla, on='motif')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 134
epitope_to_hla_map = {
    'AVFDRKSDAK':'A*11:01',
    'IVTDFSVIK':'A*11:01',
    'SIIPSGPLK':'A*11:01',
    'ATIGTAMYK':'A*11:01',
    'NLVPMVATV':'A*02:01',
    'GLCTLVAML':'A*02:01',
    'GILGFVFTL':'A*02:01',
    'FMYSDFHFI':'A*02:01',
    'CLGGLLTMV':'A*02:01',
    'RPPIFIRRL':'B*07:02',
    'TPRVTGGGAM':'B*07:02',
    'LPFDKTTVM':'B*07:02'
}

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 135
# use BEAM data where available
mapped = df_m_disease["majority_beam_epitope"].map(epitope_to_hla_map)
# df_m_disease["MHC_I_restricted_allele"] = mapped.combine_first(df_m_disease["MHC_I_restricted_allele"])
df_m_disease["MHC_I_restricted_allele"] = df_m_disease["MHC_I_restricted_allele"].where(
    mapped.isna(),
    mapped
)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 138
df_m_disease.to_csv('data/disease_associated_motifs_hla.csv')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 139
df_m_disease = pd.read_csv('data/disease_associated_motifs_hla.csv', index_col=0)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 144
motif_disease_highlights = [37, 39, 40, 44, 84,563,61, 984, 303]

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 148
plt.figure(figsize=(2,4))
sns.heatmap(
    df_m_disease.loc[df_m_disease.motif.isin(motif_disease_highlights),['motif','n_cells','n_donors','n_studies']].set_index('motif').sort_index(), 
    vmax=50, cmap='Blues', annot=True, cbar=False, fmt='.5g', linecolor='w', linewidths=1) # cbar_kws={'label':'Count','location':'top'})
plt.yticks(ticks=[])
plt.ylabel('')
plt.savefig(savedir+'selected_motifs_ncells_donors_studies.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 151
for motif in [37, 39, 40, 44, 84,563,61, 984, 303]:
    cell2tcr.draw_cdr3(df[df.motif==motif], savefig_title=savedir+f'motif_{motif}.pdf', put_title=False)
#     print(df_m_disease[df_m_disease.motif==motif].predicted_pathogen)
#     display(df_m_disease[df_m_disease.motif==motif])

## 8. Disease-associated motifs on atlas UMAPs

Generate saved atlas-level and compartment-specific UMAP panels for selected motifs.

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 154
import scanpy as sc
import numpy as np

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 155
# load T cell atlas data
obs = pd.read_csv(
    root_dir + 'datasets/pan_infection_atlas/snakemake_toolbox/out/checkpoint_objects/checkpoint_6/obs.csv', 
    index_col=0,
)
obs.head()

# GSE259231 needs to be split. We got the data set through personal communication for GSE259231 (HBV) and GSE150305 (HCV = re-used data from a previous publication)
obs.loc[obs.pathogen=='HCV', 'study'] = 'GSE150305'
adata = sc.AnnData(obs=obs)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 156
UMAP = np.load(root_dir+'datasets/pan_infection_atlas/snakemake_toolbox/out/checkpoint_objects/checkpoint_6/UMAPs/all_T_cells_umap_neighbors_30_mindist_13_spread_1.npy')
# UMAP = np.load('/nfs/team205/tcr_pipeline/datasets/pan_infection_atlas/snakemake_toolbox/out/checkpoint_objects/checkpoint_6/scANVI_improved_val/X_umap.npy')
adata.obsm["X_umap"] = UMAP
adata

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 157
adata = adata[adata.obs_names.isin(df.index)]

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 158
adata.obs['motif'] = df.loc[adata.obs_names, 'motif'].values

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 163
motif_to_pathogen_dict = {
    '563':'HIV',
    '25':'CMV',
    '61':'HPV',
    '71':'SARS-CoV-2 (CD8)',
    '865':'EBV',
    '49':'Influenza',
    '324':'SARS-CoV-2 (CD4 CTL)',
    '1062':'SARS-CoV-2 (CD4 Activated)',
    '41':'SARS-CoV-2 (CD4 CM)',
}

adata.obs['motif'] = adata.obs['motif'].astype(str)
adata.obs['motif_named'] = adata.obs['motif'].map(motif_to_pathogen_dict)


color_named_map = {
    'CMV':'cornflowerblue',
    'Influenza':'tomato',
    'HPV':'lightgreen',
    'EBV':'yellow',
    'SARS-CoV-2 (CD8)':'darkred',
    'SARS-CoV-2 (CD4 CTL)':'purple',
    'SARS-CoV-2 (CD4 Activated)':'darkgreen',
    'SARS-CoV-2 (CD4 CM)':'darkgoldenrod',
    'HIV':'black'
}

cats = adata.obs['motif_named'].astype('category').cat.categories
palette = [color_named_map.get(c, 'whitesmoke') for c in cats]

g = sc.pl.umap(
    adata,
    color='motif_named',
    groups=motif_to_pathogen_dict.values(),
    frameon=False,
    palette=palette,
    size=10,
    na_color='gainsboro',
    na_in_legend=False,
    return_fig=True,
    show=False,
#     rasterized=True
)
g.savefig(savedir+'umap_motifs')
g.show()

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 170
adata_cd8 = sc.read_h5ad(root_dir+'datasets/pan_infection_atlas/snakemake_toolbox/out/checkpoint_objects/checkpoint_6/adata_CD8.h5ad')
adata_cd8

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 171
adata = adata_cd8

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 172
adata = adata[adata.obs_names.isin(df.index)]

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 173
adata.obs['motif'] = df.loc[adata.obs_names, 'motif'].values

In [ ]:
df_m_disease = pd.read_csv(DATA_DIR / "disease_associated_motifs_hla.csv", index_col=0)
df_m_disease_idx = df_m_disease.set_index("motif", drop=False)


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 184
motifs = [
    40, # sars CD8 activated
    39, # flu CD8 CM
    85, # flu cd8 activated
    51, # EBV CD8 EM
    402, # EBV cd8 CM
    984, # tuberculosis CD8 EF
    565, # SARS cd8 EF
    312, # SARS CD4 CTL (mhc2 restricted)
    526, # SARS CD4 activated CTL 
    61, # HPV CD8 EX,
    563, # HIV CD8 PEX
#     239, # SARS cd8 ex
#     800, # sars PEX
    105, # VZV CD8 RM
    952, # CMV cd8 RM
    1177, # flu cd8 RM
    322, # sars cd8 RM
]
adata.obs["motif"] = adata.obs["motif"].astype(str)

fig, axes = plt.subplots(ncols=3, nrows=5, figsize=(12,15))
for m, ax in zip(motifs, axes.flatten()):
    m = str(m)
#     print(m, df_m_disease_idx.loc[int(m),['predicted_pathogen','majority_annotation']].values)

    # make a per-iteration obs column: motif id for matching cells, else NA
    col = "motif_one"
    adata.obs[col] = adata.obs["motif"].where(adata.obs["motif"].eq(m))

    # categorical so groups works and legend is sane
    adata.obs[col] = adata.obs[col].astype("category")

    sc.pl.embedding(
        adata,
        basis="umap3013",
        color=col,
        groups=[m],
        frameon=False,
        title=f"No. {m}: {df_m_disease_idx.loc[int(m),['predicted_pathogen']].values[0]}, {df_m_disease_idx.loc[int(m),['majority_annotation']].values[0]}",
        legend_loc=None,
        size=10,
        na_color="gainsboro",
        na_in_legend=False,
        show=False,
        ax=ax,
    )
fig.savefig(savedir+'umap_motifs_cd4_cd8_new')
fig.savefig(savedir+'umap_motifs_cd4_cd8_new.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 185
adata = sc.read_h5ad(root_dir+'datasets/pan_infection_atlas/snakemake_toolbox/out/checkpoint_objects/checkpoint_6/adata_CD4.h5ad')
adata

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 190
motifs = [
    49, # flu CD4 CM ?
    312, # SARS CD4 CTL 
    465, # SARS CD4 CTL activated 
    269, # SARS CD4 activated 
    923, # SARS CD4 PH 
    621, # SARS CD4 EF 
    328, # SARS CD4 activated 
    324, # SARS CD4 CTL 
    121, # EBV CD4 CM 
    145, # yellow fever CD4 CM
    677, # HIV CD4 CTL ?
    526, # SARS CD4 activated CTL
]
adata.obs["motif"] = adata.obs["motif"].astype(str)

fig, axes = plt.subplots(ncols=3, nrows=5, figsize=(12,15))
for m, ax in zip(motifs, axes.flatten()):
    m = str(m)
#     print(m, df_m_disease_idx.loc[int(m),['predicted_pathogen','majority_annotation']].values)

    # make a per-iteration obs column: motif id for matching cells, else NA
    col = "motif_one"
    adata.obs[col] = adata.obs["motif"].where(adata.obs["motif"].eq(m))
    # categorical so groups works and legend is sane
    adata.obs[col] = adata.obs[col].astype("category")

    sc.pl.embedding(
        adata,
        basis="umap1513",
        color=col,
        groups=[m],
        frameon=False,
        title=f"No. {m}: {df_m_disease_idx.loc[int(m),['predicted_pathogen']].values[0]}, {df_m_disease_idx.loc[int(m),['majority_annotation']].values[0]}",
        legend_loc=None,
        size=10,
        na_color="gainsboro",
        na_in_legend=False,
        show=False,
        ax=ax,
    )
fig.savefig(savedir+'umap_motifs_cd4')
fig.savefig(savedir+'umap_motifs_cd4.pdf')

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 197, plotting removed, prep retained
motif_to_pathogen_dict = {
    '563':'HIV',
    '25':'CMV',
    '61':'HPV',
    '71':'SARS-CoV-2',
    '865':'EBV',
    '49':'Influenza',
#     '324':'SARS-CoV-2 (CD4 CTL)',
#     '1062':'SARS-CoV-2 (CD4 Activated)',
#     '41':'SARS-CoV-2 (CD4 CM)',
}

adata.obs['motif'] = adata.obs['motif'].astype(str)
adata.obs['motif_named'] = adata.obs['motif'].map(motif_to_pathogen_dict)


color_named_map = {
    'CMV':'cornflowerblue',
    'Influenza':'tomato',
    'HPV':'lightgreen',
    'EBV':'yellow',
    'SARS-CoV-2':'darkred',
#     'SARS-CoV-2 (CD4 CTL)':'purple',
#     'SARS-CoV-2 (CD4 Activated)':'darkgreen',
#     'SARS-CoV-2 (CD4 CM)':'darkgoldenrod',
    'HIV':'black'
}

cats = adata.obs['motif_named'].astype('category').cat.categories
palette = [color_named_map.get(c, 'whitesmoke') for c in cats]


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 198
for motif in motif_to_pathogen_dict.values():
    g = sc.pl.embedding(
        adata,
        basis='umap3013',
        color='motif_named',
        groups=motif,
        frameon=False,
        palette=palette,
        size=10,
        na_color='gainsboro',
        na_in_legend=False,
        return_fig=True,
        show=False,
        title=motif,
    #     figsize=(5,5),
    #     rasterized=True
    )
    g.savefig(savedir+f'umap_motifs_{motif}.pdf')
    g.show()

## 9. Motif composition across T cell states

Generate saved CD4 and CD8 motif-by-cell-state heatmaps for disease-associated motifs.

In [ ]:
df_m_disease = pd.read_csv(DATA_DIR / "disease_associated_motifs_hla.csv", index_col=0)
disease_motifs = df_m_disease["motif"].unique()
motif_pathogen_map = df_m_disease[["motif", "predicted_pathogen"]].set_index("motif")["predicted_pathogen"].to_dict()


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 207
df_ = df_sub[df_sub.motif.isin(df_m_disease[
    (df_m_disease.majority_annotation.str.contains('CD4'))&
    (df_m_disease.majority_annotation_all_cells_frac>0.5)].motif.unique())&(df_sub.annotation_level_1=='T CD4')]
# df_ = df_sub[df_sub.motif.isin(disease_motifs)&(df_sub.annotation_level_1=='T CD4')]
# df_ = df[df.motif.isin(disease_motifs)&(df.annotation_level_1=='T CD4')]

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 208
df_plot = pd.crosstab(df_["motif"], df_["annotation_level_3"])
df_plot['predicted_pathogen'] = df_plot.index.map(motif_pathogen_map)

state_cols = df_plot.columns.drop("predicted_pathogen")

# sum counts across motifs within each predicted_pathogen
agg_by_pathogen = (
    df_plot[df_plot.predicted_pathogen.notna()]
    .groupby(["predicted_pathogen",'motif'], dropna=False)[state_cols]
    .sum()
    .sort_values(state_cols[0], ascending=False)  # optional sort
)

agg_by_pathogen

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 209
agg_std = agg_by_pathogen.div(agg_by_pathogen.sum(axis=1), axis=0)
agg_std

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 210
pathogen_colors = {
    "CMV": "#393b79",
    "Dengue_virus": "#5254a3",
    "EBV": "#6b6ecf",
    "HBV": "#637939",
    "HCV": "#8ca252",
    "HIV": "#b5cf6b",
    "HPV": "#8c6d31",
#     "HSV-2": "#bd9e39",
    "Yellow fever": "#bd9e39",
    "Healthy": "#e7ba52",
    "Influenza_CMV_EBV": "#843c39",
    "Influenza virus": "#ad494a",
    "Influenza": "#ad494a",
    "Plasmodium_falciparum": "#d6616b",
    "SARS-CoV-2": "#7b4173",
    "SARS-CoV-1": "tomato",
    "Tuberculosis": "#a55194",
    "VZV": "#ce6dbd",
    "other": "#de9ed6",
}

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 211
import seaborn as sns
import matplotlib.patches as mpatches

In [ ]:
order_cd4 = list(reversed(agg_std.sum().sort_values()[-15:].index))
order_cd4


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 213
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

# palette
pathogen_colors = {
    "CMV": "#393b79",
    "Dengue_virus": "#5254a3",
    "EBV": "#6b6ecf",
    "HBV": "#637939",
    "HCV": "#8ca252",
    "HIV": "#b5cf6b",
    "Yellow fever": "#bd9e39",
    "HPV": "#8c6d31",
    "Healthy": "#e7ba52",
    "Influenza_CMV_EBV": "#843c39",
    "Influenza virus": "#ad494a",
    "Influenza": "#ad494a",
    "Plasmodium_falciparum": "#d6616b",
    "SARS-CoV-1": "tomato",
    "SARS-CoV-2": "#7b4173",
    "Tuberculosis": "#a55194",
    "VZV": "#ce6dbd",
    "other": "#de9ed6",
}

X_ = agg_std
order_cols = order_cd4

# keep groups in order
# groups = [(p, g) for p, g in X_.groupby(level=0, sort=False)]
# pathos = [p for p, _ in groups]
pathos = sorted(X_.index.get_level_values(0).unique(), key=str.lower, reverse=True)
groups = [(p, X_.xs(p, level=0, drop_level=False)) for p in pathos]

# height ratios proportional to number of rows (cap optional)
row_counts = [max(1, g.shape[0]) for _, g in groups]

gap = 2. # gap between heatmaps of different pathogens
n = len(groups)

height_ratios = []
for i, (_, g) in enumerate(groups):
    height_ratios.append(max(1, g.shape[0]))
    if i < n - 1:
        height_ratios.append(gap)

fig = plt.figure(figsize=(6, 0.03 * sum(height_ratios) + 1))
gs = fig.add_gridspec(
    nrows=len(height_ratios), ncols=2,
    width_ratios=[0.12, 0.95],
    height_ratios=height_ratios,
    wspace=0.02, hspace=0.0   # important: no fractional spacing
)

axes = np.empty((n, 2), dtype=object)
r = 0
for i in range(n):
    axes[i, 0] = fig.add_subplot(gs[r, 0])
    axes[i, 1] = fig.add_subplot(gs[r, 1], sharex=axes[0, 1] if i else None)
    r += 2  # skip the gap row

# one shared colorbar (figure coords)
cbar_ax = fig.add_axes([-0.1, 0.25, 0.01, 0.2])

# legend LUT (only those present)
lut_all = {k: v for k, v in pathogen_colors.items() if k in pd.Index(pathos)}

for i, (patho, X) in enumerate(groups):
    ax_color = axes[i, 0]
    ax_heat = axes[i, 1]

#     if X.shape[0] <= 1:
#         ax_color.axis("off")
#         ax_heat.axis("off")
#         continue

    X_plot = X.droplevel("predicted_pathogen")
    M = X_plot.loc[:, order_cols]
    n_rows = M.shape[0]
    # row clustering only
    if n_rows <= 1:
        row_order = np.arange(n_rows)  # [] or [0]
    else:
        Z = linkage(pdist(M.values), method="average")
        row_order = leaves_list(Z)
#     Z = linkage(pdist(M.values), method="average")
#     row_order = leaves_list(Z)
    M_ord = M.iloc[row_order]

    # row colors (Series so .iloc works)
    pathogen = X.index.get_level_values("predicted_pathogen")
    row_colors = pd.Series(
        pathogen.map(pathogen_colors).fillna(pathogen_colors.get("other", "#cccccc")),
        index=X_plot.index,
        name="row_color",
    )
    row_colors_ord = row_colors.iloc[row_order]

    # draw strip (n x 1 x 3)
    strip_rgb = np.array([to_rgb(c) for c in row_colors_ord.to_numpy()], dtype=float)
    ax_color.imshow(strip_rgb[:, None, :], aspect="auto")
    ax_color.set_xticks([])
    ax_color.set_yticks([])
    for sp in ax_color.spines.values():
        sp.set_visible(False)

    sns.heatmap(
        M_ord,
        ax=ax_heat,
        cmap="Reds",
        yticklabels=False,
        xticklabels = False, # if i < len(groups) - 1 else True,
        cbar=(i == 0),
        cbar_ax=cbar_ax if i == 0 else None,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Proportion of cells in state"} if i == 0 else None,
    )
    ax_heat.set_ylabel("")
#     ax_heat.set_xlabel("" if i < len(groups) - 1 else ax_heat.get_xlabel())
    ax_heat.set_xlabel("")
    ax_heat.tick_params(axis="x", bottom=False, labelbottom=False) 

    ax_heat.set_xticks(np.arange(M_ord.shape[1] + 1), minor=True)
    ax_heat.grid(which="minor", axis="x", linewidth=3, color="white")
    ax_heat.tick_params(which="minor", bottom=False)
#     break
# legend once
handles = [mpatches.Patch(color=lut_all[k], label=k) for k in lut_all]
axes[0, 1].legend(
    handles=handles,
    title="predicted_pathogen",
    bbox_to_anchor=(-0.6, 1.02),
    loc="upper left",
    borderaxespad=0,
)

# turn x ticks on only for the last heatmap
last_ax = axes[len(groups) - 1, 1]
last_ax.tick_params(axis="x", bottom=True, labelbottom=True)

# put ticks at cell centers and label with your column names
cols = list(order_cols)
last_ax.set_xticks(np.arange(len(cols)) + 0.5)
last_ax.set_xticklabels(cols, rotation=90, ha="center")
fig.savefig(
    savedir+"motifs_by_cellstate_cd4.pdf",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.3,
    facecolor="white"
)
plt.show()


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 215
# df_ = df[df.motif.isin(disease_motifs)&(df.annotation_level_1=='T CD8')]
# df_ = df_sub[df_sub.motif.isin(disease_motifs)&(df_sub.annotation_level_1=='T CD8')]
df_ = df_sub[df_sub.motif.isin(df_m_disease[
    (df_m_disease.majority_annotation.str.contains('CD8'))&
    (df_m_disease.majority_annotation_all_cells_frac>0.5)].motif.unique())&(df_sub.annotation_level_1=='T CD8')]

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 216
df_plot = pd.crosstab(df_["motif"], df_["annotation_level_3"])
df_plot['predicted_pathogen'] = df_plot.index.map(motif_pathogen_map)

state_cols = df_plot.columns.drop("predicted_pathogen")

# sum counts across motifs within each predicted_pathogen
agg_by_pathogen = (
    df_plot[df_plot.predicted_pathogen.notna()]
    .groupby(["predicted_pathogen",'motif'], dropna=False)[state_cols]
    .sum()
    .sort_values(state_cols[0], ascending=False)  # optional sort
)

agg_by_pathogen

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 217
agg_std = agg_by_pathogen.div(agg_by_pathogen.sum(axis=1), axis=0)
agg_std

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 218
pathogen_colors = {
    "CMV": "#393b79",
    "Dengue_virus": "#5254a3",
    "EBV": "#6b6ecf",
    "HBV": "#637939",
    "HCV": "#8ca252",
    "HIV": "#b5cf6b",
    "HPV": "#8c6d31",
#     "HSV-2": "#bd9e39",
    "Yellow fever": "#bd9e39",
    "Healthy": "#e7ba52",
    "Influenza_CMV_EBV": "#843c39",
    "Influenza virus": "#ad494a",
    "Influenza": "#ad494a",
    "Plasmodium_falciparum": "#d6616b",
    "SARS-CoV-2": "#7b4173",
    "SARS-CoV-1": "tomato",
    "Tuberculosis": "#a55194",
    "VZV": "#ce6dbd",
    "other": "#de9ed6",
}

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 219
import seaborn as sns
import matplotlib.patches as mpatches

In [ ]:
order_cd8 = list(reversed(agg_std.sum().sort_values().index))[:-3]
order_cd8


In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 221
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

# palette
pathogen_colors = {
    "CMV": "#393b79",
    "Dengue_virus": "#5254a3",
    "EBV": "#6b6ecf",
    "HBV": "#637939",
    "HCV": "#8ca252",
    "HIV": "#b5cf6b",
    "Yellow fever": "#bd9e39",
    "HPV": "#8c6d31",
    "Healthy": "#e7ba52",
    "Influenza_CMV_EBV": "#843c39",
    "Influenza virus": "#ad494a",
    "Influenza": "#ad494a",
    "Plasmodium_falciparum": "#d6616b",
    "SARS-CoV-1": "tomato",
    "SARS-CoV-2": "#7b4173",
    "Tuberculosis": "#a55194",
    "VZV": "#ce6dbd",
    "other": "#de9ed6",
}

X_ = agg_std
order_cols = order_cd8

# keep groups in order
# groups = [(p, g) for p, g in X_.groupby(level=0, sort=False)]
# pathos = [p for p, _ in groups]
pathos = sorted(X_.index.get_level_values(0).unique(), key=str.lower, reverse=True)
groups = [(p, X_.xs(p, level=0, drop_level=False)) for p in pathos]

# height ratios proportional to number of rows (cap optional)
row_counts = [max(1, g.shape[0]) for _, g in groups]

gap = 2. # gap between heatmaps of different pathogens
n = len(groups)

height_ratios = []
for i, (_, g) in enumerate(groups):
    height_ratios.append(max(1, g.shape[0]))
    if i < n - 1:
        height_ratios.append(gap)

fig = plt.figure(figsize=(5, 0.02 * sum(height_ratios) + 1.5))
gs = fig.add_gridspec(
    nrows=len(height_ratios), ncols=2,
    width_ratios=[0.12, 0.95],
    height_ratios=height_ratios,
    wspace=0.02, hspace=0.0   # important: no fractional spacing
)

axes = np.empty((n, 2), dtype=object)
r = 0
for i in range(n):
    axes[i, 0] = fig.add_subplot(gs[r, 0])
    axes[i, 1] = fig.add_subplot(gs[r, 1], sharex=axes[0, 1] if i else None)
    r += 2  # skip the gap row

# one shared colorbar (figure coords)
cbar_ax = fig.add_axes([-0.1, 0.25, 0.01, 0.2])

# legend LUT (only those present)
lut_all = {k: v for k, v in pathogen_colors.items() if k in pd.Index(pathos)}

for i, (patho, X) in enumerate(groups):
    ax_color = axes[i, 0]
    ax_heat = axes[i, 1]

#     if X.shape[0] <= 1:
#         ax_color.axis("off")
#         ax_heat.axis("off")
#         continue

    X_plot = X.droplevel("predicted_pathogen")
    M = X_plot.loc[:, order_cols]
    n_rows = M.shape[0]
    # row clustering only
    if n_rows <= 1:
        row_order = np.arange(n_rows)  # [] or [0]
    else:
        Z = linkage(pdist(M.values), method="average")
        row_order = leaves_list(Z)
#     Z = linkage(pdist(M.values), method="average")
#     row_order = leaves_list(Z)
    M_ord = M.iloc[row_order]

    # row colors (Series so .iloc works)
    pathogen = X.index.get_level_values("predicted_pathogen")
    row_colors = pd.Series(
        pathogen.map(pathogen_colors).fillna(pathogen_colors.get("other", "#cccccc")),
        index=X_plot.index,
        name="row_color",
    )
    row_colors_ord = row_colors.iloc[row_order]

    # draw strip (n x 1 x 3)
    strip_rgb = np.array([to_rgb(c) for c in row_colors_ord.to_numpy()], dtype=float)
    ax_color.imshow(strip_rgb[:, None, :], aspect="auto")
    ax_color.set_xticks([])
    ax_color.set_yticks([])
    for sp in ax_color.spines.values():
        sp.set_visible(False)

    sns.heatmap(
        M_ord,
        ax=ax_heat,
        cmap="Reds",
        yticklabels=False,
        xticklabels = False, # if i < len(groups) - 1 else True,
        cbar=(i == 0),
        cbar_ax=cbar_ax if i == 0 else None,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Proportion of cells in state"} if i == 0 else None,
    )
    ax_heat.set_ylabel("")
#     ax_heat.set_xlabel("" if i < len(groups) - 1 else ax_heat.get_xlabel())
    ax_heat.set_xlabel("")
    ax_heat.tick_params(axis="x", bottom=False, labelbottom=False) 

    ax_heat.set_xticks(np.arange(M_ord.shape[1] + 1), minor=True)
    ax_heat.grid(which="minor", axis="x", linewidth=3, color="white")
    ax_heat.tick_params(which="minor", bottom=False)
    
# legend once
handles = [mpatches.Patch(color=lut_all[k], label=k) for k in lut_all]
axes[0, 1].legend(
    handles=handles,
    title="predicted_pathogen",
    bbox_to_anchor=(-0.6, 1.02),
    loc="upper left",
    borderaxespad=0,
)

# turn x ticks on only for the last heatmap
last_ax = axes[len(groups) - 1, 1]
last_ax.tick_params(axis="x", bottom=True, labelbottom=True)

# put ticks at cell centers and label with your column names
cols = list(order_cols)
last_ax.set_xticks(np.arange(len(cols)) + 0.5)
last_ax.set_xticklabels(cols, rotation=90, ha="center")
fig.savefig(
    savedir+"motifs_by_cellstate_cd8.pdf",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.3,
    facecolor="white"
)
plt.show()


## 10. TCR motifs by predicted pathogen and annotation source

Generate the summary scatter plot of disease-associated TCR motifs by pathogen, donor count, and annotation source.

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 228
df_m_disease.fillna('nan', inplace=True)

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 229
sorted_pathogens = [
    'CMV',
    'EBV',
    'VZV',
    'Influenza',
    'SARS-CoV-2',
    'HPV',
    'HIV',
    'Dengue_virus',
    'Yellow fever',
    'SARS-CoV-1',
    'Tuberculosis',
    'other',
]

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 230
df_m_disease.loc[df_m_disease.predicted_pathogen_source=='beam', 'predicted_pathogen_source'] = 'multimer_binding'
df_m_disease.loc[df_m_disease.predicted_pathogen=='HPV', 'predicted_pathogen_source'] = 'multimer_binding'  # override source for HPV motifs as we have multimer validation for these

In [ ]:
# Source: pan_infection_atlas_tcr_similarity.ipynb cell 232
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Combined plot (no CD4/CD8 split) — exclude motifs with pathogen == 'nan' and add jitter
df_plot = df_m_disease[~df_m_disease.majority_pathogen.eq('nan')].reset_index(drop=True)
# pathogen_counts = df_plot['predicted_pathogen'].value_counts()
# sorted_pathogens = pathogen_counts.index.tolist()

def add_jitter_to_categorical_ordered(data, col, sorted_categories, jitter_amount=0.1):
    cat_to_num = {cat: i for i, cat in enumerate(sorted_categories)}
    x_numeric = data[col].map(cat_to_num).values
    x_jittered = x_numeric + np.random.normal(0, jitter_amount, len(data))
    return x_jittered, cat_to_num

def add_y_jitter(y_series, jitter_amount=0.15):
    y = y_series.values if hasattr(y_series, 'values') else np.array(y_series)
    return y + np.random.normal(0, jitter_amount, len(y))

fig, ax = plt.subplots(figsize=(8,4))
x_jitter, _ = add_jitter_to_categorical_ordered(df_plot, 'predicted_pathogen', sorted_pathogens, jitter_amount=0.1)
y_jittered = add_y_jitter(df_plot['n_donors'], jitter_amount=0.15)
# norm = matplotlib.colors.Normalize(vmin=0, vmax=100)  # cap at 1000
scatter = ax.scatter(
    x_jitter,
    y_jittered,
    c=df_plot['predicted_pathogen_source'].map({'multimer_binding': 'skyblue', 'vdjdb': 'orange', 'majority_pathogen': 'maroon'}).fillna('gray'),
    s=200,
    alpha=0.6,
    # cmap='jet',
#     norm=norm,
    edgecolors='black',
    linewidth=0,
)
ax.set_xlabel('Predicted pathogen', fontsize=12)
ax.set_ylabel('Number of donors', fontsize=12)
ax.set_title(f'All TCR motifs (n={len(df_plot)})', fontsize=13)
ax.set_yscale('log')
ax.set_xticks(range(len(sorted_pathogens)))
ax.set_xticklabels(sorted_pathogens, rotation=50, ha='center')
# ax.set_ylim(bottom=1, top=120)
ax.set_ylim(top=120)
# legend with the 3 annotation sources
handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='skyblue', markersize=10, label='Multimer binding'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='maroon', markersize=10, label='Exclusive infection history'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=10, label='VDJdb'),
    # plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, label='Unknown source'),
]
ax.legend(handles=handles, title='Pathogen evidence type', loc='upper right')
# cbar = plt.colorbar(scatter, ax=ax)
# cbar.set_label('Number of studies', fontsize=11)
plt.tight_layout()
# plt.savefig(savedir+'disease_motifs_predicted_pathogen_vs_n_donors')
# plt.savefig(savedir+'disease_motifs_predicted_pathogen_vs_n_donors.pdf')
plt.show()
